# YOLO ile Görüntü Analizi: İnsan ve Araç Tespitinden Trafik Yoğunluğu Ölçümü

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Bu aşamada Ultralytics kütüphanesi Colab ortamına kurulmuştur. Ultralytics, YOLO modellerini yüklemeyi, eğitmeyi ve görüntüler üzerinde çalıştırmayı kolaylaştıran bir Python paketidir. Görünen 8.4.115 değeri YOLO modelinin değil, kurulan Ultralytics paketinin sürümüdür. Bu aşamada henüz bir YOLO modeli seçilmemiştir.

In [ ]:
!pip install -U ultralytics

Bu aşamada COCO veri kümesiyle önceden eğitilmiş YOLO11 Nano modeli yüklenmektedir. Model, insan, otomobil ve otobüs gibi günlük nesneleri sınıf ve konum bilgileriyle tespit edebilir. Model adındaki n, Nano sürümünü ifade eder. Nano model küçük ve hızlı olduğu için ilk denemeler ve Jetson gibi uç cihazlar için uygundur.

| Veri kümesi          | İçeriği                                                    | Kullanım alanı                                         |
| -------------------- | ---------------------------------------------------------- | ------------------------------------------------------ |
| **Objects365**       | 365 farklı nesne kategorisi                                | Çok çeşitli günlük nesneleri tespit etme               |
| **Open Images**      | Çok çeşitli nesnelerin bulunduğu geniş görüntü koleksiyonu | Genel nesne tespiti                                    |
| **KITTI**            | Araç, yaya ve bisikletli içeren sürüş görüntüleri          | Trafik ve otonom sürüş                                 |
| **VisDrone**         | Drone ile çekilmiş görüntülerde araçlar ve insanlar        | Yukarıdan görüntüleme, küçük nesne tespiti             |
| **SKU-110K**         | Raflarda yoğun biçimde yerleştirilmiş ürünler              | Mağaza ve raf analizi                                  |
| **Kendi veri kümen** | Senin çekip etiketlediğin görüntüler                       | Vida türü, çizik veya üretim hatası gibi özel görevler |


yolo11n.pt	Nano	En küçük ve hızlı

yolo11s.pt	Small	Biraz daha doğru, daha yavaş

yolo11m.pt	Medium	Daha güçlü, daha ağır

yolo11l.pt	Large	Yüksek hesaplama ihtiyacı

yolo11x.pt	Extra Large	En ağır sürüm


In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")

print("YOLO11 Nano modeli yüklendi.")

Bu aşamada nesne tespitinde kullanılacak test görseli internetten Colab ortamına indirilmektedir. Görüntüde insanlar ve bir otobüs bulunduğu için modelin aynı görüntüde birden fazla nesneyi bulması gözlemlenebilecektir. Bu aşamada YOLO henüz tahmin yapmamaktadır; yalnızca giriş görüntüsü hazırlanmaktadır.

Burada:

wget: Görseli internetten indirir.

-O test_resmi.jpg: Dosyayı bu adla kaydeder.

display(...): Görseli Colab ekranında gösterir.

width=600: Yalnızca ekrandaki gösterim genişliğidir; görüntü dosyasını değiştirmez.

Çalıştırdığında kutu çizilmemiş otobüs görseli görünmelidir. Görüntü göründüğünde sonraki aşamada YOLO’ya tahmin yaptıracağız.

In [ ]:
from IPython.display import Image, display

!wget -q https://ultralytics.com/images/bus.jpg -O test_resmi.jpg

display(Image(filename="test_resmi.jpg", width=600))

Bu aşamada test görüntüsü YOLO11 Nano modeline verilmektedir. Model görüntüyü inceleyerek COCO veri kümesinden öğrendiği nesneleri bulur. Her nesne için sınıf adı, sınırlayıcı kutu ve güven oranı üretir. Model burada yeniden eğitilmemekte, daha önce öğrendiği bilgilerle tahmin yapmaktadır. Bu işleme çıkarım (inference) denir.

Buradaki:

source: İncelenecek görüntünün dosya yolu

conf=0.25: Güven oranı en az %25 olan tespitleri kabul eder.

sonuclar[0]: İlk görüntünün tespit sonuçları

save(...): Kutuların çizildiği sonucu kaydeder.

display(...): Sonuç görüntüsünü ekranda gösterir.

Çalıştırınca insanların ve otobüsün çevresinde kutular, kutuların üzerinde sınıf adları ve güven oranları görünmelidir.

In [ ]:
sonuclar = model.predict(
    source="test_resmi.jpg",
    conf=0.25
)

sonuclar[0].save(filename="tespit_sonucu.jpg")

display(Image(filename="tespit_sonucu.jpg", width=600))

Uygulama doğru çalıştı. YOLO görüntüde:

4 person, 1 bus

tespit etti.

YOLO yalnızca görüntünün üzerine kutu çizmez. Her tespit için nesnenin sınıfını, güven skorunu ve kutunun köşe koordinatlarını sayısal olarak üretir. Bu bilgiler daha sonra nesne sayma, konum belirleme veya AI Agent karar sisteminde kullanılabilir.

In [ ]:
for kutu in sonuclar[0].boxes:

    sinif_id = int(kutu.cls[0])
    sinif_adi = model.names[sinif_id]
    guven = float(kutu.conf[0])

    x1, y1, x2, y2 = map(int, kutu.xyxy[0].tolist())

    print(
        "Nesne:", sinif_adi,
        "| Güven:", round(guven, 2),
        "| Koordinatlar:", x1, y1, x2, y2
    )

Nesne: bus | Güven: 0.94 | Koordinatlar: 12 198 563 546

Otobüs kutusu, resimde sol üstte (12,198) noktasından başlayıp sağ altta (563,546) noktasına kadar uzanıyor.



---




# Şimdi örnek resmi bırakıp bilgisayardan istediğimiz görüntüyü seçebileceğimiz hâle getiriyoruz.

Bu aşamada kullanıcı bilgisayarından bir görüntü seçerek Colab ortamına yükler. Yüklenen görüntünün dosya adı alınır ve sonraki aşamada YOLO modeline giriş olarak verilir. Böylece uygulama yalnızca sabit test görseliyle değil, kullanıcının seçtiği farklı görüntülerle çalışabilir.

Trafik görselini Colab’a indirme

Bilgi notu

Bu aşamada otomobil, otobüs, motosiklet ve insan gibi farklı nesneler içeren kalabalık bir trafik görüntüsü Colab’a indirilmektedir. Görüntüde bazı nesneler birbirini kapattığı veya uzakta kaldığı için modelin gerçek trafik koşullarındaki davranışı gözlemlenebilir.

Görsel, Colab’ın geçici çalışma alanına şu konumda indi:


/content/trafik.jpg

In [ ]:
!wget -q -O trafik.jpg "https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcRabpgA7dwodH-fOqcgWJxPy5AXW3HhWL8a7xmxo2KLMy_AeoeXRcOColM&s=10"

resim_yolu = "trafik.jpg"

display(Image(filename=resim_yolu, width=700))

In [ ]:
import os

print("Dosya var mı?", os.path.exists("/content/trafik.jpg"))
print("Dosyanın konumu:", "/content/trafik.jpg")

Beklenen çıktı:

Dosya var mı? True
Dosyanın konumu: /content/trafik.jpg

Ardından sol taraftaki dosya panelinde:

Google Drive klasörlerinden üst dizine çık.

Dosya panelini yenile.

/content altında trafik.jpg dosyasını ara.

In [ ]:
from google.colab import files

yuklenen_dosyalar = files.upload()

resim_yolu = next(iter(yuklenen_dosyalar))

print("Seçilen görsel:", resim_yolu)

Bu aşamada trafik görüntüsü YOLO11 Nano modeline verilmektedir. Model, görüntüdeki insan, otomobil, motosiklet, otobüs ve benzeri COCO sınıflarını arar. Her tespit için sınırlayıcı kutu, sınıf adı ve güven skoru üretir.

In [ ]:
trafik_sonuclari = model.predict(
    source="/content/trafik.jpg",
    conf=0.25,
    imgsz=960
)

trafik_sonuclari[0].save(filename="/content/trafik_sonucu.jpg")

display(Image(filename="/content/trafik_sonucu.jpg", width=800))

Burada:

conf=0.25: Güveni %25’in altında kalan tahminleri göstermez.

imgsz=960: Görüntüyü daha yüksek çözünürlükte işler. Uzak ve küçük nesnelerin bulunmasına yardımcı olabilir.

trafik_sonucu.jpg: Kutular çizilmiş sonuç görüntüsüdür.

Bulunan nesneler

Üst satıra göre:

İngilizce sınıf	Türkçesi	Sayı

person	  İnsan	          10

bicycle	  Bisiklet 	       1

car	      Otomobil	       7

motorcycle	Motosiklet	   3

truck	    Kamyon	         3

traffic light	Trafik ışığı 1

Model toplam:

10+1+7+3+3+1=25

nesne kutusu üretmiş.

Gradio, Python ile geliştirilen yapay zekâ modellerine tarayıcı üzerinden kullanılabilen basit bir arayüz ekler. Kullanıcı arayüzden bir görüntü yükler; görüntü YOLO modeline gönderilir ve kutulu sonuç ekranda gösterilir. Gradio nesne tespitini kendisi yapmaz, yalnızca kullanıcı ile YOLO modeli arasında arayüz görevi görür.

In [ ]:
!pip install -U gradio

In [ ]:
import gradio as gr

print("Gradio başarıyla yüklendi.")
print("Gradio sürümü:", gr.__version__)

Gradio, Python dilinde geliştirdiğiniz makine öğrenmesi modelleri veya veri bilimi uygulamaları için hızlıca web tabanlı kullanıcı arayüzleri (UI) oluşturmanızı sağlayan açık kaynaklı bir Python kütüphanesidir.

Bu aşamada Gradio arayüzüne yüklenen görüntü YOLO11 Nano modeline gönderilir. Model nesneleri tespit eder, kutulu görüntüyü oluşturur ve bulunan nesnelerin sayılarını Türkçe olarak listeler. Böylece önceki ayrı kod hücreleri tek bir kullanıcı arayüzünde birleştirilmiş olur.

Çalıştırıldığında bir Gradio arayüzü ve geçici bağlantı oluşacaktır:

Görsel yükleme alanına tıkla.

Bir trafik görseli seç.

Submit/Gönder düğmesine bas.

Sağ tarafta kutulu görüntü ve nesne sayıları gösterilir.

In [ ]:
from collections import Counter
import gradio as gr

turkce_adlar = {
    "person": "İnsan",
    "bicycle": "Bisiklet",
    "car": "Otomobil",
    "motorcycle": "Motosiklet",
    "bus": "Otobüs",
    "truck": "Kamyon",
    "traffic light": "Trafik ışığı",
    "cat": "Kedi",
    "dog": "Köpek",
    "bird": "Kuş"
}


def nesneleri_tani(resim_yolu):

    if resim_yolu is None:
        return None, "Lütfen bir görüntü yükleyin."

    sonuc = model.predict(
        source=resim_yolu,
        conf=0.25,
        imgsz=960,
        verbose=False
    )[0]

    # YOLO'nun kutu çizdiği görüntü
    kutulu_goruntu = sonuc.plot()

    # BGR renk düzenini RGB'ye çevir
    kutulu_goruntu = kutulu_goruntu[:, :, ::-1]

    # Bulunan sınıfları say
    sinif_numaralari = sonuc.boxes.cls.int().tolist()
    sinif_adlari = [
        model.names[numara]
        for numara in sinif_numaralari
    ]

    sayim = Counter(sinif_adlari)

    if not sayim:
        ozet = "Nesne tespit edilemedi."
    else:
        satirlar = ["TESPİT EDİLEN NESNELER"]

        for sinif_adi, adet in sayim.items():
            turkce_ad = turkce_adlar.get(sinif_adi, sinif_adi)
            satirlar.append(f"{turkce_ad}: {adet}")

        ozet = "\n".join(satirlar)

    return kutulu_goruntu, ozet


uygulama = gr.Interface(
    fn=nesneleri_tani,

    inputs=gr.Image(
        type="filepath",
        label="Görüntü yükleyin"
    ),

    outputs=[
        gr.Image(label="YOLO nesne tespit sonucu"),
        gr.Textbox(label="Nesne sayımı", lines=10)
    ],

    title="YOLO11 Görüntüde Nesne Tanıma",
    description=(
        "Bir görüntü yükleyin. YOLO11 Nano modeli "
        "görüntüdeki nesneleri bulup kutularla gösterecektir."
    )
)

uygulama.launch(share=True)



---



---



---



# **VİDEO ÜZERİNDE TESPİT İŞLEMLERİ**



---



---



---


# **Videoyu Colab’a yükleme**

Bu aşamada nesne tespiti yapılacak video Colab ortamına yüklenmektedir. YOLO videoyu doğrudan tek parça olarak incelemez; videodaki kareleri sırayla işleyerek her karedeki nesneleri ayrı ayrı tespit eder.

Videoyu /content klasörüne indirme
Bu aşamada hazır bir trafik videosu internetten Colab’ın geçici çalışma alanına indirilmektedir. Video henüz YOLO tarafından işlenmemekte, yalnızca giriş verisi olarak hazırlanmaktadır.

In [ ]:
!wget -O /content/trafik_video.mp4 \
"https://raw.githubusercontent.com/intel-iot-devkit/sample-videos/master/person-bicycle-car-detection.mp4"

In [ ]:
import os

video_yolu = "/content/trafik_video.mp4"

print("Video var mı?", os.path.exists(video_yolu))
print("Video boyutu:", round(os.path.getsize(video_yolu) / 1024 / 1024, 2), "MB")
print("Video yolu:", video_yolu)

In [ ]:
from IPython.display import Video, display

display(Video(video_yolu, embed=True, width=600))

Bu aşamada YOLO, videodaki kareleri sırayla işler. Her karede nesnelerin sınıfını, konumunu ve güven skorunu tahmin eder. Kutular işlenmiş karelerin üzerine çizilerek yeni bir sonuç videosu oluşturulur. Bu işlem henüz nesne takibi değildir; aynı araç farklı karelerde yeniden tespit edilir.

In [ ]:
video_sonuclari = model.predict(
    source=video_yolu,
    conf=0.25,
    imgsz=640,
    save=True,
    stream=True,
    project="/content/trafik_video",
    name="tespit",
    exist_ok=True,
    verbose=False
)

# stream=True sonucu kare kare üretir.
# Döngü, bütün video tamamlanıncaya kadar çalışır.
for _ in video_sonuclari:
    pass

print("Video nesne tespiti tamamlandı.")

Yukarıdaki İşlem Uzun Süreceğinden;
Colab’da GPU’yu açma

Colab menüsünden:

Çalışma zamanı menüsünü aç.

Çalışma zamanı türünü değiştir seçeneğine bas.

Donanım hızlandırıcı bölümünü T4 GPU olarak seç.

Kaydet düğmesine bas.

Colab yeniden bağlanana kadar bekle.

In [ ]:
import torch

print("CUDA kullanılabilir mi?", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU bulunamadı.")

GPU Bağlantısı Başarılı:

GPU bağlantısı başarılıysa önce şu hücreleri yeniden çalıştıracağız:

In [ ]:
!pip install -U ultralytics

In [ ]:
from ultralytics import YOLO
import torch

model = YOLO("yolo11n.pt")

print("Model hazır.")
print("Kullanılacak GPU:", torch.cuda.get_device_name(0))

Trafik Videosunu Tekrar İndiriyoruz.

In [ ]:
!wget -O /content/trafik_video.mp4 \
"https://raw.githubusercontent.com/intel-iot-devkit/sample-videos/master/person-bicycle-car-detection.mp4"

video_yolu = "/content/trafik_video.mp4"

print("Video indirildi:", video_yolu)

Yolo modelini tekrar indiriyoruz.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")

Bu aşamada video kareleri YOLO11 Nano modeli tarafından NVIDIA GPU üzerinde işlenir. device=0, bilgisayardaki ilk GPU’nun kullanılacağını belirtir. half=True ise hesaplamaları 16 bit duyarlılıkla gerçekleştirerek T4 gibi GPU’larda işlemi hızlandırır.

Aşağıdaki koddaki:

device=0: İlk NVIDIA GPU’yu kullanır.

half=True: FP16 hesaplama kullanarak GPU işlem hızını artırabilir.

stream=True: Videoyu kare kare işler; belleğin dolmasını önler.

save=True: Kutulu sonuç videosunu kaydeder.

time.perf_counter(): Toplam işlem süresini ölçer.

Video Üzerinden Tanımayı Başlatıyoruz:

In [ ]:
import time

baslangic = time.perf_counter()

video_sonuclari = model.predict(
    source=video_yolu,
    conf=0.25,
    imgsz=640,
    device=0,
    half=True,
    save=True,
    stream=True,
    project="/content/yolo_video",
    name="gpu_tespit",
    exist_ok=True,
    verbose=False
)

for _ in video_sonuclari:
    pass

bitis = time.perf_counter()

print("GPU ile nesne tespiti tamamlandı.")
print("Toplam süre:", round(bitis - baslangic, 2), "saniye")
print("Kayıt klasörü: /content/yolo_video/gpu_tespit")

İşlem başarıyla tamamlanmış:

Toplam süre: 23.78 saniye

Results saved to /content/yolo_video/gpu_tespit

Sonuç videosunu bulma ve oynatma

YOLO tarafından işlenen video, seçilen çıktı klasörüne kaydedilmiştir. Video tarayıcıda sorunsuz oynatılabilmesi için H.264 biçimine dönüştürülerek Colab ekranında gösterilecektir.

In [ ]:
from pathlib import Path

klasor = Path("/content/yolo_video/gpu_tespit")

video_dosyalari = (
    list(klasor.glob("*.mp4")) +
    list(klasor.glob("*.avi")) +
    list(klasor.glob("*.mov"))
)

print("Bulunan sonuçlar:", video_dosyalari)

sonuc_video_yolu = str(video_dosyalari[0])
print("Sonuç videosu:", sonuc_video_yolu)

Şimdi AVI videosunu Colab tarayıcısında oynatılabilecek H.264 MP4 biçimine dönüştürüyoruz.

In [ ]:
!ffmpeg -y -loglevel error \
-i "/content/yolo_video/gpu_tespit/trafik_video.avi" \
-vcodec libx264 \
-pix_fmt yuv420p \
-an \
"/content/yolo_video_sonuc.mp4"

print("MP4 dönüşümü tamamlandı.")

Oynatma Kodları:

In [ ]:
from IPython.display import Video, display

display(
    Video(
        "/content/yolo_video_sonuc.mp4",
        embed=True,
        width=800
    )
)

Ancak önce sonuç videosunu indir. Çünkü çalışma ortamını CPU’ya geçirince /content klasörü sıfırlanabilir. Ve video kaybolabilir.

Sonuç videosunu indir:

In [ ]:
from google.colab import files

files.download("/content/yolo_video_sonuc.mp4")

İndirme tamamlandıktan sonra:

Çalışma zamanı

Çalışma zamanı türünü değiştir

Donanım hızlandırıcı → Yok/None

Kaydet

Bunun sonucunda:

Colab CPU ortamına geçer.

GPU kullanımın sona erer.

Kod hücrelerin ve Drive’daki not defterin kalır.

/content içindeki video, model ve geçici dosyalar silinebilir.

Daha sonra GPU’ya döndüğünde kurulum ve indirme hücrelerini yeniden çalıştırman gerekir.

Kısa bilgi notu:

GPU yalnızca yoğun model eğitimi ve video işleme sırasında kullanılmalıdır. GPU gerektirmeyen kod yazma, veri inceleme ve açıklama aşamalarında CPU ortamına dönmek, Colab kaynaklarının gereksiz tüketilmesini önler.

CPU ortamına geçtiğini doğrulamak için

In [ ]:
import torch

print("CUDA kullanılabilir mi?", torch.cuda.is_available())

if torch.cuda.is_available():
    print("Aktif ortam: GPU")
    print("GPU modeli:", torch.cuda.get_device_name(0))
else:
    print("Aktif ortam: CPU")
    print("GPU bağlantısı bulunmuyor.")

In [ ]:
from google.colab import files

yuklenen_dosyalar = files.upload()

video_yolu = next(iter(yuklenen_dosyalar))

print("Video Colab'a yüklendi.")
print("Video yolu:", "/content/" + video_yolu)

In [ ]:
from IPython.display import Video, display

display(
    Video(
        "/content/" + video_yolu,
        embed=True,
        width=800
    )
)



---



---



---



## **Proje Adı: YOLO11s ve ByteTrack ile Trafik Yoğunluğu Ölçümü Projesi**

# ***Aşama 1 — Projenin amacı ve genel işlem akışı***

Proje Bilgi:

  Bu projede, sabit kamera ile kaydedilmiş bir trafik videosundaki araçların ve yol yoğunluğunun belirlenmesi amaçlanmıştır. COCO veri kümesiyle önceden eğitilmiş YOLO11s modeli kullanılarak otomobil, motosiklet, otobüs ve kamyonlar tespit edilmiş; ByteTrack algoritmasıyla araçlar video boyunca takip edilmiştir.

  Kamera görüntüsündeki yollar ayrı bölgelere ayrılmış ve araç kutularının her yol üzerinde kapladığı alan hesaplanmıştır. Elde edilen doluluk oranlarına göre yollar düşük yoğunlukta yeşil, orta yoğunlukta sarı ve yüksek yoğunlukta kırmızı renkle gösterilmiştir. Sonuç videosunda araç kutuları, takip numaraları, yol doluluk oranları ve yoğunluk seviyeleri birlikte sunulmuştur.

Bu uygulama gelecekteki trafik yoğunluğunu tahmin etmemekte, videoda görülen mevcut trafik yoğunluğunu ölçmektedir.

# ***Aşama 2 — Colab GPU ortamının kontrol edilmesi***

# ***Aşama 3 - T4 GPU’nun etkinleştirilmesi***

# ***Aşama 4 - Ultralytics kütüphanesinin kurulması***

# ***Aşama 5 - YOLO11s modelinin yüklenmesi***

# ***Aşama 6 - Trafik videosunun Colab’a indirilmesi***

# ***Aşama 7 - Video özelliklerinin incelenmesi***

# ***Aşama 8 - Trafik videosunun oynatılması***

# ***Aşama 9 - Videodan örnek kare çıkarılması***

# ***Aşama 10 - Örnek kareden alınan görselin gösterilmesi***

# ***Aşama 11 - Yolo ile araç tespiti yapılması***

# ***Aşama 12 - Yolo ile tespit edilen araç sınıflarının ve sayılarının gösterilmesi***

# ***Aşama 13 — Örnek Karede Tespit Edilen Araçların Sayılması***

# ***Aşama 14 — ByteTrack ile Araçların Video Boyunca Takip Edilmesi***

# ***Aşama 15 — Araç Takip Sonuç Videosunun Bulunması***

# ***Aşama 16 — Videoyu H.264 MP4 biçimine dönüştürme***

# ***Aşama 17 — Takip videosunu oynatma***

#***Aşama 18 — Yoğunluk Hesaplama Yönteminin Belirlenmesi ve Yol Seçimine Hazırlık***

# ***Aşama 19 — Ana Yatay Yol Bölgesinin Oluşturulması***

# ***Aşama 20 — Seçilen Yol Bölgesinin Yarı Saydam Yeşil Renklendirilmesi***

# ***Aşama 21 — Seçilen Yol Bölgesinin Görüntülenmesi ve Kontrol Edilmesi***

# ***Aşama 22 — Araçların Yol Üzerinde Kapladığı Alanın Hesaplanması***

# ***Aşama 23 — Her Video Karesi İçin Yol Doluluk Oranının Hesaplanması***

# ***Aşama 24 — Doluluk Değerlerinin Tabloya Eklenmesi ve Hareketli Ortalamanın Hesaplanması***

# ***Aşama 25 — Yolun En Boş, En Yoğun ve Ortalama Doluluk Değerlerinin Hesaplanması***

# ***Aşama 26 — Trafik Yoğunluğu Eşiklerinin Belirlenmesi***

# ***Aşama 27 — Doluluk Değerlerinin Trafik Seviyesine Dönüştürülmesi***

# ***Aşama 28 — Trafik Yoğunluğu Sınıflarının Tabloya Eklenmesi***

# ***Aşama 29 — Trafik Yoğunluğu Bilgilerinin Renkli Video Üzerinde Gösterilmesi***

# ***Aşama 30 — Sonuç Videosunun Tarayıcıda Açılabilecek H.264 Biçimine Dönüştürülmesi***

# ***Aşama 31 — Hazırlanan Trafik Yoğunluğu Videosunun Görüntülenmesi***

# ***Aşama 32 — Dört Yol Bölgesinin Ayrı Ayrı Trafik Yoğunluğu Analizi***


Önemli düzeltme: Colab’daki eski “yüzde 33 ve yüzde 66’ya göre göreli eşik belirleme” bölümü, son uygulamada kullanılan ortak %5 ve %10 eşikleriyle çelişiyor. Nihai anlatımda eski eşik yöntemini deneme aşaması olarak göstermeli veya kaldırmalısın. Aksi hâlde uygulamanın hangi yöntemi kullandığı belirsiz olur.

# ***Aşama 2 — GPU ortamının kontrol edilmesi***


Trafik videosu çok sayıda görüntü karesinden oluştuğu için nesne tespiti işlemleri GPU üzerinde gerçekleştirilecektir. Bu aşamada PyTorch kullanılarak Colab ortamında CUDA destekli NVIDIA GPU bulunup bulunmadığı kontrol edilmiştir. GPU, YOLO modelinin matris hesaplamalarını CPU’ya göre daha hızlı gerçekleştirir.

Bu aşamada Google Colab ortamında GPU’nun kullanılabilir olup olmadığı kontrol edilmiştir. CPU bilgisayarın genel işlemlerini yürütürken GPU, çok sayıda görüntü ve matematiksel işlemi paralel olarak gerçekleştirebilir. Bu nedenle YOLO modelinin videodaki yüzlerce kareyi analiz etmesi GPU üzerinde CPU’ya göre daha hızlıdır.

CPU; videoyu okuma, verileri düzenleme ve sonuç videosunu oluşturma işlemlerinde kullanılır. GPU ise YOLO11s modelinin araçları tespit etme hesaplamalarını hızlandırır. Colab ortamında NVIDIA Tesla T4 GPU etkinleştirilerek nesne tespit işlemleri bu donanım üzerinde gerçekleştirilmiştir.

In [ ]:
import torch

gpu_var_mi = torch.cuda.is_available()

print("CUDA kullanılabilir mi?", gpu_var_mi)

if gpu_var_mi:
    print("GPU modeli:", torch.cuda.get_device_name(0))
    print("Kullanılacak cihaz: cuda:0")
else:
    print("GPU bulunamadı.")
    print("Kullanılacak cihaz: cpu")

Zaten mevcut durumda GPU ya geçmediğimiz için GPU bulunamadı çıkması normal.

# ***Aşama 3 - T4 GPU’nun etkinleştirilmesi***

Colab menüsünden:

Çalışma zamanı menüsünü aç.

Çalışma zamanı türünü değiştir seçeneğine bas.

Donanım hızlandırıcı bölümünden T4 GPU seç.

Kaydet düğmesine bas.

Colab yeniden bağlanana kadar bekle.


Önemli: Çalışma zamanı değiştiğinde /content klasörü sıfırlanabilir. Ancak henüz yeni trafik projesinin dosyalarını hazırlamadığımız için şu anda sorun oluşturmaz.

In [ ]:
import torch

gpu_var_mi = torch.cuda.is_available()

print("CUDA kullanılabilir mi?", gpu_var_mi)

if gpu_var_mi:
    print("GPU modeli:", torch.cuda.get_device_name(0))
    print("Kullanılacak cihaz: cuda:0")
else:
    print("GPU bulunamadı.")

Üstteki Kodları kullanarak Tesla T4 - GPU modeline bağlandık.

# ***Aşama 4 - Ultralytics kütüphanesinin kurulması***

Bu aşamada YOLO modellerini Python üzerinden kullanmayı sağlayan Ultralytics kütüphanesi kurulmuştur. Daha sonra COCO veri kümesinin 80 nesne sınıfıyla önceden eğitilmiş YOLO11s modeli yüklenmiştir. Model sıfırdan eğitilmemekte; otomobil, motosiklet, otobüs ve kamyon gibi daha önce öğrendiği sınıflar kullanılmaktadır.

In [ ]:
!pip install -U ultralytics

Buradaki sürüm, YOLO modelinin değil, Ultralytics Python paketinin sürümüdür.

# ***Aşama 5 - YOLO11s modelinin yüklenmesi***

Aşağıdaki kod COCO ile eğitilmiş YOLO11 Small modelini indirir ve belleğe yükler.

In [ ]:
from ultralytics import YOLO
import ultralytics

model = YOLO("yolo11s.pt")

print("Ultralytics sürümü:", ultralytics.__version__)
print("Yüklenen model: YOLO11 Small")
print("Model dosyası: yolo11s.pt")

In [ ]:
Yolo modeli başarıyla indirildi.

# ***Aşama 6 - Trafik videosunun Colab’a indirilmesi***

Bu aşamada sabit kamerayla çekilmiş bir kavşak videosu Colab ortamına indirilmiştir. Video analizinden önce çözünürlük, saniyedeki kare sayısı, toplam kare sayısı ve video süresi belirlenmiştir. YOLO modeli videoyu tek parça olarak değil, görüntü karelerini sırayla işleyerek analiz edecektir.

In [ ]:
!wget -O /content/trafik_akisi.mp4 \
"https://raw.githubusercontent.com/antonmilev/TrafficDetection/main/vid.mp4"

Buradaki:

wget: Dosyayı internetten indirir.

-O: İndirilen dosyanın adını ve konumunu belirler.

/content/trafik_akisi.mp4: Videonun Colab’daki tam yoludur.

# ***Aşama 7 - Video özelliklerinin incelenmesi***

In [ ]:
import cv2
import os

video_yolu = "/content/trafik_akisi.mp4"

video = cv2.VideoCapture(video_yolu)

fps = video.get(cv2.CAP_PROP_FPS)
kare_sayisi = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
genislik = int(video.get(cv2.CAP_PROP_FRAME_WIDTH))
yukseklik = int(video.get(cv2.CAP_PROP_FRAME_HEIGHT))
sure = kare_sayisi / fps

video.release()

print("Video bulundu mu?", os.path.exists(video_yolu))
print("Çözünürlük:", genislik, "x", yukseklik)
print("FPS:", fps)
print("Toplam kare:", kare_sayisi)
print("Video süresi:", round(sure, 2), "saniye")

Çözünürlük	Her karenin piksel boyutu

FPS	Bir saniyedeki görüntü karesi

Kare sayısı	YOLO’nun inceleyeceği toplam görüntü sayısı

Süre	Kare sayısı ÷ FPS

Video gerçekte 10 saniye sürer ve YOLO yaklaşık 250 ayrı görüntüyü inceleyecektir.

# ***Aşama 8 - Trafik videosunun oynatılması***

Video oynatma kodu:

In [ ]:
from IPython.display import Video, display

display(
    Video(
        video_yolu,
        embed=True,
        width=800
    )
)

# ***Aşama 9 - Videodan örnek kare çıkarılması***

Bu aşamada trafik videosunun orta bölümünden bir görüntü karesi alınmıştır. YOLO11s modeli yalnızca trafikle ilişkili otomobil, motosiklet, otobüs ve kamyon sınıflarını arayacak şekilde çalıştırılmıştır. Tek kare testiyle modelin video üzerindeki nesne tespit performansı, bütün video işlenmeden önce kontrol edilmiştir.

In [ ]:
import cv2

video = cv2.VideoCapture(video_yolu)

toplam_kare = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
orta_kare_numarasi = toplam_kare // 2

video.set(cv2.CAP_PROP_POS_FRAMES, orta_kare_numarasi)

basarili, ornek_kare = video.read()

video.release()

if basarili:
    ornek_kare_yolu = "/content/ornek_kare.jpg"
    cv2.imwrite(ornek_kare_yolu, ornek_kare)

    print("Örnek kare çıkarıldı.")
    print("Kare numarası:", orta_kare_numarasi)
    print("Dosya yolu:", ornek_kare_yolu)
else:
    print("Video karesi okunamadı.")

# ***Aşama 10 - Örnek kareden alınan görselin gösterilmesi***

In [ ]:
from IPython.display import Image, display

display(
    Image(
        filename="/content/ornek_kare.jpg",
        width=800
    )
)

İnsan, trafik ışığı veya başka nesneler bu deneyde gösterilmez. Çünkü yoğunluk hesabımız şimdilik motorlu taşıtlara dayanacaktır.

# ***Aşama 11 - Yolo ile araç tespiti yapılması***

In [ ]:
ornek_sonuc = model.predict(
    source="/content/ornek_kare.jpg",
    classes=[2, 3, 5, 7],
    conf=0.25,
    imgsz=960,
    device=0,
    quantize=16,
    verbose=False
)[0]

ornek_sonuc.save(
    filename="/content/ornek_kare_sonuc.jpg"
)

print("Tek kare araç tespiti tamamlandı.")

# ***Aşama 12 - Yolo ile tespit edilen araç sınıflarının ve sayılarının gösterilmesi***

In [ ]:
display(
    Image(
        filename="/content/ornek_kare_sonuc.jpg",
        width=800
    )
)

# ***Aşama 13 — Örnek Karede Tespit Edilen Araçların Sayılması***

Bu aşamada YOLO11s modelinin örnek görüntü karesinde tespit ettiği araçların sınıf numaraları alınmıştır. Sınıf numaraları otomobil, motosiklet, otobüs ve kamyon adlarına dönüştürülmüş; Counter kullanılarak her araç türünün sayısı hesaplanmıştır. Örnek karede 24 otomobil ve 1 otobüs olmak üzere toplam 25 motorlu araç tespit edilmiştir.

In [ ]:
from collections import Counter

sinif_numaralari = ornek_sonuc.boxes.cls.int().tolist()
sinif_adlari = [
    model.names[numara]
    for numara in sinif_numaralari
]

sayim = Counter(sinif_adlari)

turkce_adlar = {
    "car": "Otomobil",
    "motorcycle": "Motosiklet",
    "bus": "Otobüs",
    "truck": "Kamyon"
}

print("ÖRNEK KARE ARAÇ SAYIMI")

for sinif_adi, adet in sayim.items():
    print(turkce_adlar[sinif_adi], ":", adet)

print("Toplam motorlu araç:", sum(sayim.values()))

# ***Aşama 14 — ByteTrack ile Araçların Video Boyunca Takip Edilmesi***

YOLO her karede araçları bulur fakat aynı otomobilin önceki karedeki otomobil olduğunu tek başına bilmez. ByteTrack, araçlara kimlik numarası vererek aynı aracı ardışık karelerde takip eder.

Bu aşamada YOLO11s tarafından tespit edilen araçlar ByteTrack algoritmasıyla video boyunca takip edilmiştir. Takip sistemi her araca bir kimlik numarası atayarak aynı aracın ardışık görüntü karelerinde eşleştirilmesini sağlar. Her karedeki aktif araç sayısı, araç kutuları ve takip kimlikleri daha sonraki trafik yoğunluğu hesabında kullanılmak üzere kaydedilmiştir.

5.1. Takip işlemini başlatma

In [ ]:
import time

baslangic = time.perf_counter()

benzersiz_idler = set()
kare_kayitlari = []

takip_sonuclari = model.track(
    source=video_yolu,
    tracker="bytetrack.yaml",
    persist=True,
    classes=[2, 3, 5, 7],
    conf=0.25,
    imgsz=960,
    device=0,
    quantize=16,
    stream=True,
    save=True,
    project="/content/trafik_projesi",
    name="takip_sonucu",
    exist_ok=True,
    verbose=False
)

for kare_numarasi, sonuc in enumerate(takip_sonuclari):

    if sonuc.boxes.id is not None:

        takip_idleri = (
            sonuc.boxes.id
            .int()
            .cpu()
            .tolist()
        )

        siniflar = (
            sonuc.boxes.cls
            .int()
            .cpu()
            .tolist()
        )

        kutular = (
            sonuc.boxes.xyxy
            .cpu()
            .tolist()
        )

        benzersiz_idler.update(takip_idleri)

    else:
        takip_idleri = []
        siniflar = []
        kutular = []

    kare_kayitlari.append({
        "kare": kare_numarasi,
        "aktif_arac": len(takip_idleri),
        "takip_idleri": takip_idleri,
        "siniflar": siniflar,
        "kutular": kutular
    })

bitis = time.perf_counter()

print("Araç takibi tamamlandı.")
print("İşlenen kare sayısı:", len(kare_kayitlari))
print("Üretilen farklı takip kimliği:", len(benzersiz_idler))
print("Toplam işlem süresi:", round(bitis - baslangic, 2), "saniye")
print("Sonuç klasörü: /content/trafik_projesi/takip_sonucu")

# ***Aşama 15 — Araç Takip Sonuç Videosunun Bulunması***

Bu aşamada YOLO11s ve ByteTrack tarafından oluşturulan takip videosu çıktı klasöründe bulunmuştur. Ultralytics tarafından üretilen AVI videosu, Colab tarayıcısıyla uyumlu olması için H.264 kodlamalı MP4 biçimine dönüştürülmüş ve ekranda oynatılmıştır. Videodaki takip kimlikleri, araçların ardışık karelerde aynı nesne olarak eşleştirilmesini göstermektedir.

In [ ]:
from pathlib import Path

sonuc_klasoru = Path(
    "/content/trafik_projesi/takip_sonucu"
)

video_dosyalari = (
    list(sonuc_klasoru.glob("*.avi")) +
    list(sonuc_klasoru.glob("*.mp4")) +
    list(sonuc_klasoru.glob("*.mov"))
)

print("Bulunan video dosyaları:")

for dosya in video_dosyalari:
    print(dosya)

takip_video_yolu = str(video_dosyalari[0])

print("Kullanılacak video:", takip_video_yolu)

# ***Aşama 16 — Videoyu H.264 MP4 biçimine dönüştürme***

In [ ]:
!ffmpeg -y -loglevel error \
-i "{takip_video_yolu}" \
-vcodec libx264 \
-pix_fmt yuv420p \
-an \
"/content/trafik_takip_sonucu.mp4"

print("Video dönüşümü tamamlandı.")

Buradaki:

libx264: H.264 video kodlaması yapar.

yuv420p: Tarayıcılarla uyumlu renk biçimi kullanır.

-an: Ses kanalını kaldırır.

-y: Aynı isimli dosya varsa yeniden oluşturur.

Bu dönüşüm, YOLO tespitlerini değiştirmez. Yalnızca videonun sıkıştırma biçimini değiştirir.

# ***Aşama 17 — Takip videosunu oynatma***

In [ ]:
from IPython.display import Video, display

display(
    Video(
        "/content/trafik_takip_sonucu.mp4",
        embed=True,
        width=900
    )
)

# ***Aşama 18 — Yoğunluk Hesaplama Yönteminin Belirlenmesi ve Yol Seçimine Hazırlık***

Kimlik numarası değişmiyorsa takip başarılıdır.

Yolun üzerine yarı saydam bir renk katmanı ekleriz:

Yeşil: Trafik düşük

Sarı: Trafik orta

Kırmızı: Trafik yüksek

Yoğunluk nasıl hesaplanacak?

Yalnızca araç sayısını kullanmak yeterli değildir. Şu oranı hesaplayacağız:

Doluluk oranı= Yol uzerinde arac​ların kapladıg​ı alan / Toplam yol alanı×100

Yol alanı             = 400.000 piksel

Araçların kapladığı alan = 120.000 piksel

120000/400000×100=%30

Sonuç örneği:

Doluluk	Renk	  Durum

Düşük	  Yeşil	  Akıcı

Orta	  Sarı	  Yoğunlaşıyor

Yüksek	Kırmızı	Yoğun

Yol üzerinde gösterim

In [ ]:
import cv2
import matplotlib.pyplot as plt

kare = cv2.imread("/content/ornek_kare.jpg")

if kare is None:
    print("Hata: ornek_kare.jpg bulunamadı.")
else:
    kare_rgb = cv2.cvtColor(
        kare,
        cv2.COLOR_BGR2RGB
    )

    plt.figure(figsize=(14, 8))
    plt.imshow(kare_rgb)
    plt.title("Yol bölgesi seçilecek örnek kare")
    plt.axis("off")
    plt.show()

# ***Aşama 19 — Ana Yatay Yol Bölgesinin Oluşturulması***



Bu aşamada trafik yoğunluğunun ölçüleceği ana yol bölgesi bir çokgenle tanımlanmıştır. Yol bölgesi görüntü boyutuna oranlı koordinatlarla seçilmiş ve seçimin doğruluğunu kontrol etmek amacıyla yarı saydam yeşil renkle gösterilmiştir. Görüntünün yol dışındaki bölümleri yoğunluk hesabına alınmayacaktır.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Örnek görüntüyü yeniden oku
kare = cv2.imread("/content/ornek_kare.jpg")

if kare is None:
    raise FileNotFoundError(
        "ornek_kare.jpg bulunamadı."
    )

yukseklik, genislik = kare.shape[:2]

print("Görüntü boyutu:",
      genislik,
      "x",
      yukseklik)

# Ana yatay yol bölgesi
yol_cokgeni = np.array([
    [0, int(yukseklik * 0.43)],
    [genislik - 1, int(yukseklik * 0.43)],
    [genislik - 1, int(yukseklik * 0.66)],
    [0, int(yukseklik * 0.59)]
], dtype=np.int32)

print("Yol köşe koordinatları:")
print(yol_cokgeni)

# ***Aşama 20 — Seçilen Yol Bölgesinin Yarı Saydam Yeşil Renklendirilmesi***

In [ ]:
renkli_katman = kare.copy()

yesil = (0, 255, 0)  # OpenCV: BGR

cv2.fillPoly(
    renkli_katman,
    [yol_cokgeni],
    yesil
)

renkli_kare = cv2.addWeighted(
    renkli_katman,
    0.30,
    kare,
    0.70,
    0
)

# Yol sınırını daha belirgin çiz
cv2.polylines(
    renkli_kare,
    [yol_cokgeni],
    isClosed=True,
    color=(0, 255, 0),
    thickness=4
)

# ***Aşama 21 — Seçilen Yol Bölgesinin Görüntülenmesi ve Kontrol Edilmesi***


Yeşil alan:

Ana yatay yolu kapsamalı,

Üstteki binaları kapsamamalı,

Alttaki ağaçlık alanı kapsamamalı,

Yol üzerinde hareket eden araçların bulunduğu bölgeyi içine almalıdır.

Bu aşamada yeşil renk “trafik düşük” anlamına gelmiyor. Yalnızca seçilen yol bölgesini kontrol ediyoruz.

In [ ]:
renkli_kare_rgb = cv2.cvtColor(
    renkli_kare,
    cv2.COLOR_BGR2RGB
)

plt.figure(figsize=(14, 8))
plt.imshow(renkli_kare_rgb)
plt.title("Trafik yoğunluğu ölçülecek ana yol bölgesi")
plt.axis("off")
plt.show()

# ***Aşama 22 — Araçların Yol Üzerinde Kapladığı Alanın Hesaplanması***


Hesap:

Doluluk yuzdesi=Arac​ kutularının yol icindeki birles​ik alanı/Yol Bölgesinin Alanı *100

# ***Aşama 22-a — Yol Maskesinin Oluşturulması***

Bu aşamada seçilen yol bölgesi bir ikili maske hâline getirilmiştir. Her video karesinde YOLO tarafından bulunan araç kutuları başka bir maske üzerine çizilmiş ve bu iki maskenin kesişimi hesaplanmıştır. Üst üste gelen araç kutularının aynı pikselleri birden fazla kez sayılmadan, araçların yol bölgesi içindeki birleşik doluluk oranı belirlenmiştir.

Maske, yalnızca 0 ve 255 değerlerinden oluşan siyah-beyaz bir görüntüdür:

In [ ]:
import cv2
import numpy as np

yukseklik, genislik = kare.shape[:2]

yol_maskesi = np.zeros(
    (yukseklik, genislik),
    dtype=np.uint8
)

cv2.fillPoly(
    yol_maskesi,
    [yol_cokgeni],
    255
)

yol_alani = cv2.countNonZero(yol_maskesi)

print("Yol bölgesinin alanı:",
      yol_alani,
      "piksel")

# ***Aşama 23 — Her Video Karesi İçin Yol Doluluk Oranının Hesaplanması***

In [ ]:
doluluk_oranlari = []

for kayit in kare_kayitlari:

    arac_maskesi = np.zeros(
        (yukseklik, genislik),
        dtype=np.uint8
    )

    for kutu in kayit["kutular"]:

        x1, y1, x2, y2 = map(int, kutu)

        # Koordinatları görüntü sınırları içinde tut
        x1 = max(0, min(x1, genislik - 1))
        x2 = max(0, min(x2, genislik - 1))
        y1 = max(0, min(y1, yukseklik - 1))
        y2 = max(0, min(y2, yukseklik - 1))

        cv2.rectangle(
            arac_maskesi,
            (x1, y1),
            (x2, y2),
            255,
            thickness=-1
        )

    # Yalnızca yol içinde kalan araç alanı
    yol_uzerindeki_araclar = cv2.bitwise_and(
        arac_maskesi,
        yol_maskesi
    )

    arac_alani = cv2.countNonZero(
        yol_uzerindeki_araclar
    )

    doluluk_yuzdesi = (
        arac_alani / yol_alani
    ) * 100

    doluluk_oranlari.append(
        doluluk_yuzdesi
    )

print("Hesaplanan kare sayısı:",
      len(doluluk_oranlari))
print("İlk 5 doluluk oranı:", doluluk_oranlari[:5])
print("İlk doluluk oranları:")

for oran in doluluk_oranlari[:5]:
    print(f"%{oran:.2f}")

print(f"Ortalama doluluk: %{np.mean(doluluk_oranlari):.2f}")
print("Ortalama doluluk: %", round(np.mean(doluluk_oranlari), 2))

Hesaplanan kare sayısı: 250 bir doluluk oranı değildir. Kod, videodaki 250 ayrı kare için doluluk oranı hesaplandığını bildiriyor.

Bu 250 sonuç doluluk_oranlari listesinde saklanır.

# ***Aşama 24 — Doluluk Değerlerinin Tabloya Eklenmesi ve Hareketli Ortalamanın Hesaplanması***

Bu aşamada her kare için hesaplanan yol doluluk yüzdeleri veri tablosuna eklenmiş ve ani değişimleri azaltmak için bir saniyelik hareketli ortalama hesaplanmıştır.

In [ ]:
import cv2
import pandas as pd

# Gerekli önceki sonuçları kontrol et
if "kare_kayitlari" not in globals():
    raise NameError(
        "kare_kayitlari bulunamadı. "
        "Önce Aşama 5'teki araç takip kodunu çalıştırın."
    )

if "doluluk_oranlari" not in globals():
    raise NameError(
        "doluluk_oranlari bulunamadı. "
        "Önce Aşama 9.2'deki doluluk hesaplama kodunu çalıştırın."
    )

# Kayıt sayılarının eşitliğini kontrol et
if len(kare_kayitlari) != len(doluluk_oranlari):
    raise ValueError(
        "Kare kayıtlarının sayısıyla doluluk değerlerinin "
        "sayısı eşit değil."
    )

# Veri tablosunu oluştur
grafik_verisi = pd.DataFrame({
    "kare": [
        kayit["kare"]
        for kayit in kare_kayitlari
    ],

    "aktif_arac": [
        kayit["aktif_arac"]
        for kayit in kare_kayitlari
    ]
})

# FPS değerini videodan yeniden oku
video = cv2.VideoCapture(video_yolu)
fps = video.get(cv2.CAP_PROP_FPS)
video.release()

# Kare numarasını zamana dönüştür
grafik_verisi["zaman_saniye"] = (
    grafik_verisi["kare"] / fps
)

# Bir saniyeye karşılık gelen kare sayısı
pencere_boyutu = max(
    1,
    int(round(fps))
)

print("FPS:", fps)
print(
    "Hareketli ortalama pencere boyutu:",
    pencere_boyutu,
    "kare"
)

# Doluluk sonuçlarını tabloya ekle
grafik_verisi["doluluk_yuzdesi"] = (
    doluluk_oranlari
)

# Bir saniyelik hareketli ortalama
grafik_verisi["doluluk_ortalamasi"] = (
    grafik_verisi["doluluk_yuzdesi"]
    .rolling(
        window=pencere_boyutu,
        center=True,
        min_periods=1
    )
    .mean()
)

print(
    grafik_verisi[
        [
            "zaman_saniye",
            "aktif_arac",
            "doluluk_yuzdesi",
            "doluluk_ortalamasi"
        ]
    ].head()
)

Burada 27 araç görüntünün tamamında takip ediliyor. Ancak doluluk hesabına yalnızca seçtiğimiz yatay yol bölgesi içindeki kutu parçaları giriyor. Bu nedenle aktif araç sayısı yüksekken yol doluluğu yaklaşık %3,5 olabilir.


doluluk_ortalamasi ilk karelerde daha düşük görünebilir; hareketli ortalama video başlangıcında daha az komşu kareye sahiptir.


# ***Aşama 25 — Yolun En Boş, En Yoğun ve Ortalama Doluluk Değerlerinin Hesaplanması***


Şimdi minimum, maksimum ve ortalamayı çıkarmamız gerekiyor.
Minimum, maksimum ve ortalama değerleri videodaki trafik durumunu tek bir sonuçla özetlemek için çıkarıyoruz:

Minimum doluluk: Yolun en boş olduğu anı gösterir.

Maksimum doluluk: Yolun en yoğun olduğu anı gösterir.

Ortalama doluluk: Video boyunca genel trafik seviyesini gösterir.

In [ ]:
en_dusuk_doluluk = (
    grafik_verisi["doluluk_ortalamasi"].min()
)

en_yuksek_doluluk = (
    grafik_verisi["doluluk_ortalamasi"].max()
)

ortalama_doluluk = (
    grafik_verisi["doluluk_ortalamasi"].mean()
)

print(
    "En düşük doluluk:",
    round(en_dusuk_doluluk, 2),
    "%"
)

print(
    "En yüksek doluluk:",
    round(en_yuksek_doluluk, 2),
    "%"
)

print(
    "Ortalama doluluk:",
    round(ortalama_doluluk, 2),
    "%"
)

Bu değerlere göre:

Yeşil eşik  → düşük yoğunluk

Sarı eşik   → orta yoğunluk

Kırmızı eşik → yüksek yoğunluk

sınırlarını hesaplayacağız.

# ***Aşama 26 — Trafik Yoğunluğu Eşiklerinin Belirlenmesi***


In [ ]:
alt_esik = (
    grafik_verisi["doluluk_ortalamasi"]
    .quantile(0.33)
)

ust_esik = (
    grafik_verisi["doluluk_ortalamasi"]
    .quantile(0.66)
)

print(
    "Yeşil–sarı sınırı:",
    round(alt_esik, 2),
    "%"
)

print(
    "Sarı–kırmızı sınırı:",
    round(ust_esik, 2),
    "%"
)

Aşamanın Amacı:

Bu kod, videodaki doluluk değerlerini küçükten büyüğe sıralayarak trafiği üç gruba ayıracak sınırları belirler.

Hesaplamanın Cevapladığı soru:
Minimum %2,71	Yolun en boş olduğu durumda doluluk kaçtı?

Maksimum %5,49	Yolun en yoğun olduğu durumda doluluk kaçtı?

Ortalama %3,77	Video boyunca genel doluluk kaçtı?

Alt eşik %3,15	Yeşil durum nerede bitecek?

Üst eşik %3,83	Sarı durum nerede bitecek?

Yani minimum ve maksimum, eşik hesaplamak için değil, videonun genel sonuçlarını raporlamak için bulunmuştur.

Eşikler ise her kareye renk vermek için hesaplanmıştır.

Somut olarak:

Ölçülen bütün değerlerin aralığı: %2,71 - %5,49
Genel ortalama: %3,77

Doluluk %2,71 ve altındaysa: Yeşil — düşük yoğunluk

Doluluk %2,71 - %3,77 arasındaysa: Sarı — orta yoğunluk

Doluluk %3,77’den büyükse: Kırmızı — yüksek yoğunluk

%3,15 ve altı  → Yeşil
%3,15 - %3,83 → Sarı
%3,83 üzeri   → Kırmızı
Sonuçlarımıza göre:

Önemli sınırlama: Bu renkler bu videoya göre göreceli yoğunluğu gösterir. Örneğin %3,9 gerçek hayatta yoğun trafik olmayabilir; yalnızca bu videodaki diğer karelerden daha yüksek olduğu için kırmızı olur.

Colab bilgi notu

Bu aşamada trafik yoğunluğu sınıfları, rastgele belirlenen sabit değerler yerine videodan ölçülen yol doluluk oranlarının dağılımına göre oluşturulmuştur. Doluluk değerlerinin yüzde 33’lük ve yüzde 66’lık yüzdelik noktaları eşik olarak kullanılmış; düşük yoğunluk yeşil, orta yoğunluk sarı ve yüksek yoğunluk kırmızı olarak sınıflandırılmıştır.

# ***Aşama 27 — Doluluk Değerlerinin Trafik Seviyesine Dönüştürülmesi***

Bu aşama eşikleri yeniden hesaplamaz. Önceden hesaplanan alt_esik ve ust_esik değerlerini kullanarak doluluğu sınıflandıran bir fonksiyon oluşturur:

Alt eşiğe kadar → DÜŞÜK

İki eşik arasında → ORTA

Üst eşikten sonra → YÜKSEK

In [ ]:
def yogunluk_sinifi(doluluk):

    if doluluk < alt_esik:
        return "DÜŞÜK"

    elif doluluk < ust_esik:
        return "ORTA"

    else:
        return "YÜKSEK"

# ***Aşama 28 — Trafik Yoğunluğu Sınıflarının Tabloya Eklenmesi***

In [ ]:
grafik_verisi["yogunluk"] = (
    grafik_verisi["doluluk_ortalamasi"]
    .apply(yogunluk_sinifi)
)

print(
    grafik_verisi[
        [
            "zaman_saniye",
            "aktif_arac",
            "doluluk_ortalamasi",
            "yogunluk"
        ]
    ].head(10)
)

Aşama 11 — Yolun yoğunluğa göre renklendirildiği videonun oluşturulması

Aşamanın amacı

Orijinal videoyu kare kare okuyacağız. Her karenin hesaplanan yoğunluk sınıfına göre yol bölgesini:

Düşük → Yeşil

Orta → Sarı

Yüksek → Kırmızı

renkle kaplayacağız.

Ayrıca araç kutuları, takip kimlikleri, doluluk oranı ve aktif araç sayısı gösterilecek.

Colab bilgi notu

Bu aşamada her video karesinin daha önce hesaplanan yoğunluk sınıfı okunmuştur. Ana yol bölgesi düşük yoğunlukta yeşil, orta yoğunlukta sarı ve yüksek yoğunlukta kırmızı renkle yarı saydam olarak kaplanmıştır. Araçların sınırlayıcı kutuları ve ByteTrack kimlikleri yol kaplamasının üzerine çizilmiş; doluluk oranı ve aktif araç sayısı bilgi panelinde gösterilmiştir.

# ***Aşama 29 — Trafik Yoğunluğu Bilgilerinin Renkli Video Üzerinde Gösterilmesi***

Bu aşamada her video karesinin yoğunluk sınıfı tablodan alınarak yol bölgesi düşük yoğunlukta yeşil, orta yoğunlukta sarı ve yüksek yoğunlukta kırmızı renklendirilir. Araç kutuları, takip kimlikleri, doluluk yüzdesi ve aktif araç sayısı videoya eklenerek sonuç videosu oluşturulur.

In [ ]:
import cv2

giris_video = cv2.VideoCapture(video_yolu)

video_fps = giriş_fps = giris_video.get(
    cv2.CAP_PROP_FPS
)

video_genislik = int(
    giris_video.get(cv2.CAP_PROP_FRAME_WIDTH)
)

video_yukseklik = int(
    giris_video.get(cv2.CAP_PROP_FRAME_HEIGHT)
)

gecici_video_yolu = (
    "/content/trafik_yogunluk_gecici.mp4"
)

fourcc = cv2.VideoWriter_fourcc(*"mp4v")

video_yazici = cv2.VideoWriter(
    gecici_video_yolu,
    fourcc,
    video_fps,
    (video_genislik, video_yukseklik)
)

renk_haritasi = {
    "DÜŞÜK": (0, 255, 0),     # Yeşil
    "ORTA": (0, 255, 255),    # Sarı
    "YÜKSEK": (0, 0, 255)     # Kırmızı
}

metin_haritasi = {
    "DÜŞÜK": "DUSUK",
    "ORTA": "ORTA",
    "YÜKSEK": "YUKSEK"
}

kare_numarasi = 0

while True:

    basarili, video_karesi = giris_video.read()

    if not basarili:
        break

    if kare_numarasi >= len(grafik_verisi):
        break

    tablo_satiri = grafik_verisi.iloc[
        kare_numarasi
    ]

    yogunluk = tablo_satiri["yogunluk"]

    doluluk = float(
        tablo_satiri["doluluk_ortalamasi"]
    )

    aktif_arac = int(
        tablo_satiri["aktif_arac"]
    )

    yol_rengi = renk_haritasi[yogunluk]

    # 1. Yol bölgesine renk katmanı ekle
    renkli_katman = video_karesi.copy()

    cv2.fillPoly(
        renkli_katman,
        [yol_cokgeni],
        yol_rengi
    )

    video_karesi = cv2.addWeighted(
        renkli_katman,
        0.28,
        video_karesi,
        0.72,
        0
    )

    # 2. Yol bölgesinin dış sınırını çiz
    cv2.polylines(
        video_karesi,
        [yol_cokgeni],
        isClosed=True,
        color=yol_rengi,
        thickness=5
    )

    # 3. Bu karedeki takip sonuçlarını al
    kayit = kare_kayitlari[kare_numarasi]

    kutular = kayit["kutular"]
    takip_idleri = kayit["takip_idleri"]
    siniflar = kayit["siniflar"]

    # 4. Araç kutularını ve kimliklerini çiz
    for kutu, takip_id, sinif_id in zip(
        kutular,
        takip_idleri,
        siniflar
    ):

        x1, y1, x2, y2 = map(int, kutu)

        cv2.rectangle(
            video_karesi,
            (x1, y1),
            (x2, y2),
            (255, 255, 255),
            thickness=3
        )

        sinif_adi = model.names[sinif_id]

        etiket = (
            f"{sinif_adi} ID:{takip_id}"
        )

        cv2.rectangle(
            video_karesi,
            (x1, max(0, y1 - 30)),
            (x1 + 190, y1),
            (0, 0, 0),
            thickness=-1
        )

        cv2.putText(
            video_karesi,
            etiket,
            (x1 + 5, max(20, y1 - 7)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.60,
            (255, 255, 255),
            thickness=2
        )

    # 5. Siyah yarı saydam bilgi paneli
    panel = video_karesi.copy()

    cv2.rectangle(
        panel,
        (20, 20),
        (520, 175),
        (0, 0, 0),
        thickness=-1
    )

    video_karesi = cv2.addWeighted(
        panel,
        0.60,
        video_karesi,
        0.40,
        0
    )

    # 6. Yoğunluk bilgilerini yaz
    cv2.putText(
        video_karesi,
        "TRAFIK YOGUNLUGU: "
        + metin_haritasi[yogunluk],
        (40, 70),
        cv2.FONT_HERSHEY_SIMPLEX,
        1.0,
        yol_rengi,
        thickness=3
    )

    cv2.putText(
        video_karesi,
        f"Yol doluluk: %{doluluk:.2f}",
        (40, 115),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.85,
        (255, 255, 255),
        thickness=2
    )

    cv2.putText(
        video_karesi,
        f"Aktif arac: {aktif_arac}",
        (40, 155),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.85,
        (255, 255, 255),
        thickness=2
    )

    video_yazici.write(video_karesi)

    kare_numarasi += 1

giris_video.release()
video_yazici.release()

print("Renkli yoğunluk videosu oluşturuldu.")
print("İşlenen kare sayısı:", kare_numarasi)
print("Geçici video:", gecici_video_yolu)

# ***Aşama 30 — Sonuç Videosunun Tarayıcıda Açılabilecek H.264 Biçimine Dönüştürülmesi***

In [ ]:
!ffmpeg -y -loglevel error \
-i "/content/trafik_yogunluk_gecici.mp4" \
-vcodec libx264 \
-pix_fmt yuv420p \
-an \
"/content/trafik_yogunluk_sonucu.mp4"

print("H.264 dönüşümü tamamlandı.")

# ***Aşama 31 — Hazırlanan Trafik Yoğunluğu Videosunun Görüntülenmesi***

In [ ]:
from IPython.display import Video, display

display(
    Video(
        "/content/trafik_yogunluk_sonucu.mp4",
        embed=True,
        width=900
    )
)

Aşağıdaki tek Colab hücresi bütün işlemi yapar:

YOLO11s + ByteTrack ile araçları takip eder.

Dört yol bölgesinin doluluğunu ayrı hesaplar.

Her yol için düşük–orta–yüksek eşiklerini belirler.

Yolları yeşil–sarı–kırmızı renklendirir.

Araç kutularını ve takip numaralarını çizer.

GPU ve CPU sürelerini ayrı gösterir.

Sonuç videosunu oynatır ve indirme bağlantısı oluşturur.

# ***Aşama 32 — Dört Yol Bölgesinin Ayrı Ayrı Trafik Yoğunluğu Analizi***

Bu aşamada YOLO11s araçları tespit eder, ByteTrack ise her araca bir takip numarası vererek video boyunca aynı aracı izler. Görüntüde belirlenen dört yol bölgesinin doluluk oranları ayrı ayrı hesaplanır ve her yol düşük, orta veya yüksek yoğunluk olarak sınıflandırılır. Sonuçlar yeşil, sarı ve kırmızı renklerle videoya işlenir; araç kutuları, takip numaraları ve işlem süreleri gösterilir. Son olarak analiz videosu oluşturularak oynatma ve indirme bağlantısı hazırlanır.

In [ ]:
# ============================================================
# TÜM YOLLAR İÇİN TRAFİK YOĞUNLUĞU ANALİZİ
# YOLO11s + ByteTrack + Yol Bazlı Renkli Gösterim
# ============================================================

import cv2
import time
import subprocess
import numpy as np
import pandas as pd
import torch

from pathlib import Path
from ultralytics import YOLO
from IPython.display import Video, display
from google.colab import files


# ------------------------------------------------------------
# 1. AYARLAR
# ------------------------------------------------------------

# Daha önce video_yolu değişkeni tanımlandıysa onu kullanır.
# Tanımlanmadıysa aşağıdaki yolu kullanır.
video_yolu = globals().get("video_yolu", "/content/vid.mp4")

model_yolu = "yolo11s.pt"

gecici_video_yolu = "/content/tum_yollar_gecici.mp4"
sonuc_video_yolu = "/content/tum_yollar_trafik_yogunlugu.mp4"

# COCO araç sınıfları:
# 2 = car, 3 = motorcycle, 5 = bus, 7 = truck
arac_siniflari = [2, 3, 5, 7]

guven_esigi = 0.25
goruntu_boyutu = 960

if not Path(video_yolu).exists():
    raise FileNotFoundError(
        f"Video bulunamadı: {video_yolu}\n"
        "video_yolu değişkenini doğru dosya yoluyla değiştirin."
    )

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU bulunamadı. Colab'da Çalışma zamanı > "
        "Çalışma zamanı türünü değiştir > T4 GPU seçin."
    )

print("Kullanılan GPU:", torch.cuda.get_device_name(0))
print("Video:", video_yolu)


# ------------------------------------------------------------
# 2. VİDEO BİLGİLERİNİ OKUMA
# ------------------------------------------------------------

video = cv2.VideoCapture(video_yolu)

fps = video.get(cv2.CAP_PROP_FPS)
genislik = int(video.get(cv2.CAP_PROP_FRAME_WIDTH))
yukseklik = int(video.get(cv2.CAP_PROP_FRAME_HEIGHT))
toplam_kare = int(video.get(cv2.CAP_PROP_FRAME_COUNT))

video.release()

if fps <= 0 or genislik <= 0 or yukseklik <= 0:
    raise RuntimeError("Video bilgileri okunamadı.")

print("\nVideo çözünürlüğü:", genislik, "x", yukseklik)
print("FPS:", round(fps, 2))
print("Toplam kare:", toplam_kare)


# ------------------------------------------------------------
# 3. YOL BÖLGELERİNİ TANIMLAMA
# ------------------------------------------------------------

def nokta(x_orani, y_orani):
    """
    Oransal koordinatları piksel koordinatına dönüştürür.
    Böylece farklı çözünürlüklerde aynı yol geometrisi korunur.
    """
    return [
        int(x_orani * genislik),
        int(y_orani * yukseklik)
    ]


yol_bolgeleri = {
    "SOL ANA YOL": np.array([
        nokta(0.00, 0.43),
        nokta(0.52, 0.43),
        nokta(0.52, 0.66),
        nokta(0.00, 0.66)
    ], dtype=np.int32),

    "SAG ANA YOL": np.array([
        nokta(0.52, 0.43),
        nokta(1.00, 0.43),
        nokta(1.00, 0.66),
        nokta(0.52, 0.66)
    ], dtype=np.int32),

    "UST YOL": np.array([
        nokta(0.55, 0.02),
        nokta(0.76, 0.02),
        nokta(0.59, 0.46),
        nokta(0.49, 0.46)
    ], dtype=np.int32),

    "ALT YOL": np.array([
        nokta(0.42, 0.56),
        nokta(0.58, 0.56),
        nokta(0.70, 1.00),
        nokta(0.17, 1.00)
    ], dtype=np.int32)
}


# Her yol için ayrı maske oluştur
yol_maskeleri = {}
yol_alanlari = {}

for yol_adi, cokgen in yol_bolgeleri.items():

    maske = np.zeros(
        (yukseklik, genislik),
        dtype=np.uint8
    )

    cv2.fillPoly(
        maske,
        [cokgen],
        255
    )

    yol_maskeleri[yol_adi] = maske
    yol_alanlari[yol_adi] = cv2.countNonZero(maske)


# ------------------------------------------------------------
# 4. YOLO11s + BYTETRACK İLE ARAÇ TAKİBİ
# ------------------------------------------------------------

print("\nYOLO11s ve ByteTrack analizi başladı...")

model = YOLO(model_yolu)

gpu_baslangic = time.perf_counter()

takip_sonuclari = model.track(
    source=video_yolu,
    tracker="bytetrack.yaml",
    persist=True,
    classes=arac_siniflari,
    conf=guven_esigi,
    imgsz=goruntu_boyutu,
    device=0,
    quantize=16,
    stream=True,
    verbose=False
)

kare_kayitlari = []
doluluk_kayitlari = []
benzersiz_idler = set()


for kare_numarasi, sonuc in enumerate(takip_sonuclari):

    # Araç kutuları
    if sonuc.boxes is not None and len(sonuc.boxes) > 0:

        kutular = sonuc.boxes.xyxy.cpu().numpy().astype(int)
        siniflar = sonuc.boxes.cls.cpu().numpy().astype(int)

        if sonuc.boxes.id is not None:
            takip_idleri = sonuc.boxes.id.cpu().numpy().astype(int)
        else:
            takip_idleri = np.full(len(kutular), -1)

    else:
        kutular = np.empty((0, 4), dtype=int)
        siniflar = np.empty(0, dtype=int)
        takip_idleri = np.empty(0, dtype=int)

    for takip_id in takip_idleri:
        if takip_id >= 0:
            benzersiz_idler.add(int(takip_id))

    # Bu karedeki tüm araç kutularını maskele
    arac_maskesi = np.zeros(
        (yukseklik, genislik),
        dtype=np.uint8
    )

    for kutu in kutular:

        x1, y1, x2, y2 = kutu

        x1 = np.clip(x1, 0, genislik - 1)
        x2 = np.clip(x2, 0, genislik - 1)
        y1 = np.clip(y1, 0, yukseklik - 1)
        y2 = np.clip(y2, 0, yukseklik - 1)

        cv2.rectangle(
            arac_maskesi,
            (x1, y1),
            (x2, y2),
            255,
            -1
        )

    kare_doluluklari = {}

    # Her yolun araçlarla kaplanan alanını ayrı hesapla
    for yol_adi, yol_maskesi in yol_maskeleri.items():

        yol_uzerindeki_arac_alani = cv2.bitwise_and(
            arac_maskesi,
            yol_maskesi
        )

        kaplanan_alan = cv2.countNonZero(
            yol_uzerindeki_arac_alani
        )

        doluluk_yuzdesi = (
            kaplanan_alan /
            yol_alanlari[yol_adi]
        ) * 100

        kare_doluluklari[yol_adi] = doluluk_yuzdesi

    kare_kayitlari.append({
        "kare": kare_numarasi,
        "kutular": kutular,
        "siniflar": siniflar,
        "takip_idleri": takip_idleri,
        "aktif_arac": len(kutular)
    })

    doluluk_kayitlari.append(kare_doluluklari)


gpu_suresi = time.perf_counter() - gpu_baslangic

print("Araç analizi tamamlandı.")
print("GPU analiz süresi:", round(gpu_suresi, 2), "saniye")
print("Benzersiz takip ID sayısı:", len(benzersiz_idler))


# ------------------------------------------------------------
# 5. DOLULUK VERİ TABLOSUNU OLUŞTURMA
# ------------------------------------------------------------

grafik_verisi = pd.DataFrame(doluluk_kayitlari)

grafik_verisi.insert(
    0,
    "zaman_saniye",
    np.arange(len(grafik_verisi)) / fps
)

# Yaklaşık bir saniyelik hareketli ortalama
pencere_boyutu = max(1, int(round(fps)))

for yol_adi in yol_bolgeleri:

    grafik_verisi[yol_adi + "_ORT"] = (
        grafik_verisi[yol_adi]
        .rolling(
            window=pencere_boyutu,
            center=True,
            min_periods=1
        )
        .mean()
    )


# ------------------------------------------------------------
# 6. HER YOL İÇİN YOĞUNLUK EŞİKLERİ
# ------------------------------------------------------------

yol_esikleri = {}

for yol_adi in yol_bolgeleri:

    ortalama_sutunu = yol_adi + "_ORT"

    alt_esik = grafik_verisi[ortalama_sutunu].quantile(0.33)
    ust_esik = grafik_verisi[ortalama_sutunu].quantile(0.66)

    yol_esikleri[yol_adi] = {
        "alt": alt_esik,
        "ust": ust_esik
    }

    print(
        f"{yol_adi}: "
        f"Düşük < %{alt_esik:.2f}, "
        f"Yüksek >= %{ust_esik:.2f}"
    )


def yogunluk_sinifi(deger, alt_esik, ust_esik):

    # Yol doluluk değeri bütün video boyunca neredeyse
    # değişmiyorsa yanlış sınıflandırmayı engeller.
    if abs(ust_esik - alt_esik) < 0.001:
        return "DUSUK"

    if deger < alt_esik:
        return "DUSUK"

    elif deger < ust_esik:
        return "ORTA"

    else:
        return "YUKSEK"


# OpenCV renkleri BGR sırasındadır
renk_haritasi = {
    "DUSUK":  (0, 255, 0),     # Yeşil
    "ORTA":   (0, 255, 255),   # Sarı
    "YUKSEK": (0, 0, 255)      # Kırmızı
}


# ------------------------------------------------------------
# 7. RENKLİ SONUÇ VİDEOSUNU CPU İLE OLUŞTURMA
# ------------------------------------------------------------

print("\nRenkli sonuç videosu oluşturuluyor...")

cpu_baslangic = time.perf_counter()

giris_video = cv2.VideoCapture(video_yolu)

video_yazici = cv2.VideoWriter(
    gecici_video_yolu,
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (genislik, yukseklik)
)

kare_numarasi = 0

while True:

    basarili, kare = giris_video.read()

    if not basarili:
        break

    if kare_numarasi >= len(kare_kayitlari):
        break

    kayit = kare_kayitlari[kare_numarasi]
    renkli_katman = kare.copy()

    # Her yolu kendi yoğunluğuna göre renklendir
    for yol_adi, cokgen in yol_bolgeleri.items():

        doluluk = grafik_verisi.loc[
            kare_numarasi,
            yol_adi + "_ORT"
        ]

        alt_esik = yol_esikleri[yol_adi]["alt"]
        ust_esik = yol_esikleri[yol_adi]["ust"]

        sinif = yogunluk_sinifi(
            doluluk,
            alt_esik,
            ust_esik
        )

        renk = renk_haritasi[sinif]

        cv2.fillPoly(
            renkli_katman,
            [cokgen],
            renk
        )

    # Saydam renkli katmanı gerçek kareyle birleştir
    kare = cv2.addWeighted(
        renkli_katman,
        0.28,
        kare,
        0.72,
        0
    )

    # Yol sınırlarını ve bilgilerini çiz
    yazi_y = 35

    for yol_adi, cokgen in yol_bolgeleri.items():

        doluluk = grafik_verisi.loc[
            kare_numarasi,
            yol_adi + "_ORT"
        ]

        alt_esik = yol_esikleri[yol_adi]["alt"]
        ust_esik = yol_esikleri[yol_adi]["ust"]

        sinif = yogunluk_sinifi(
            doluluk,
            alt_esik,
            ust_esik
        )

        renk = renk_haritasi[sinif]

        cv2.polylines(
            kare,
            [cokgen],
            True,
            renk,
            3
        )

        bilgi = (
            f"{yol_adi}: {sinif} - %{doluluk:.2f}"
        )

        # Yazının görünmesi için siyah arka plan
        cv2.rectangle(
            kare,
            (10, yazi_y - 24),
            (470, yazi_y + 7),
            (0, 0, 0),
            -1
        )

        cv2.putText(
            kare,
            bilgi,
            (18, yazi_y),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.65,
            renk,
            2
        )

        yazi_y += 38

    # Araç kutularını ve ByteTrack numaralarını çiz
    for kutu, takip_id, sinif_no in zip(
        kayit["kutular"],
        kayit["takip_idleri"],
        kayit["siniflar"]
    ):

        x1, y1, x2, y2 = map(int, kutu)

        cv2.rectangle(
            kare,
            (x1, y1),
            (x2, y2),
            (255, 255, 255),
            2
        )

        if takip_id >= 0:
            arac_yazisi = f"ID:{takip_id}"
        else:
            arac_yazisi = "ARAC"

        cv2.putText(
            kare,
            arac_yazisi,
            (x1, max(20, y1 - 7)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.55,
            (255, 255, 255),
            2
        )

    # Genel bilgi
    cv2.putText(
        kare,
        f"Aktif arac: {kayit['aktif_arac']}",
        (genislik - 260, 35),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (255, 255, 255),
        2
    )

    video_yazici.write(kare)

    kare_numarasi += 1


giris_video.release()
video_yazici.release()

cpu_video_suresi = time.perf_counter() - cpu_baslangic

print(
    "CPU ile renkli video oluşturma süresi:",
    round(cpu_video_suresi, 2),
    "saniye"
)


# ------------------------------------------------------------
# 8. VİDEOYU H.264 FORMATINA DÖNÜŞTÜRME
# ------------------------------------------------------------

ffmpeg_baslangic = time.perf_counter()

subprocess.run(
    [
        "ffmpeg",
        "-y",
        "-i", gecici_video_yolu,
        "-vcodec", "libx264",
        "-pix_fmt", "yuv420p",
        "-movflags", "+faststart",
        sonuc_video_yolu
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    check=True
)

ffmpeg_suresi = time.perf_counter() - ffmpeg_baslangic


# ------------------------------------------------------------
# 9. SÜRE VE SONUÇ BİLGİLERİ
# ------------------------------------------------------------

toplam_sure = (
    gpu_suresi +
    cpu_video_suresi +
    ffmpeg_suresi
)

print("\n========== İŞLEM SÜRELERİ ==========")
print("YOLO + ByteTrack analiz süresi :", round(gpu_suresi, 2), "sn")
print("CPU görüntü oluşturma süresi   :", round(cpu_video_suresi, 2), "sn")
print("FFmpeg dönüştürme süresi       :", round(ffmpeg_suresi, 2), "sn")
print("Toplam işlem süresi            :", round(toplam_sure, 2), "sn")
print("Sonuç videosu                  :", sonuc_video_yolu)


# ------------------------------------------------------------
# 10. SONUÇ VİDEOSUNU GÖSTERME
# ------------------------------------------------------------

display(
    Video(
        sonuc_video_yolu,
        embed=True,
        width=900
    )
)

# İndirmek isterseniz aşağıdaki satırın başındaki # işaretini kaldırın:
# files.download(sonuc_video_yolu)

Uygulamada sabit kamera görüntüsündeki yollar ayrı ilgi bölgeleri olarak tanımlanmıştır. YOLO11s modeli otomobil, motosiklet, otobüs ve kamyonları tespit etmiş; ByteTrack algoritması araçlara takip numarası vermiştir. Her yol bölgesinde araç kutularının kapladığı alan hesaplanarak yol bazlı doluluk yüzdesi elde edilmiştir. Doluluk değerleri videonun kendi dağılımına göre düşük, orta ve yüksek olarak sınıflandırılmış; yollar sırasıyla yeşil, sarı ve kırmızı renklerle gösterilmiştir.

Eleştirel not: Buradaki “doluluk”, gerçek araç yüzey alanı değil, YOLO sınırlayıcı kutularının yol bölgesinde kapladığı alanın yaklaşık oranıdır. Ayrıca eşikler videonun yüzde 33 ve yüzde 66 dilimlerinden üretildiği için sonuçlar mutlak trafik standardı değil, video içindeki göreli yoğunluğu gösterir.

********
Yukarıdaki videoya göre hatalı görünüp Yapılan düzeltmeler

İlk uygulamada her yolun yoğunluk eşikleri, o yolun kendi doluluk değerlerinin yüzde 33 ve yüzde 66’lık dilimlerinden üretilmişti. Bu göreli yöntem nedeniyle sağ yolda yalnızca bir araç ve %1,33 doluluk bulunmasına rağmen yol kırmızı gösteriliyordu. Bu hata giderilerek bütün yollar için ortak eşikler tanımlandı: %0–5 düşük, %5–10 orta, %10 ve üzeri yüksek. Ayrıca alt yolun çokgeni görüntüde sağa kaydığı için koordinatları sola taşındı ve yol genişliğine uygun şekilde daraltıldı.

In [ ]:
# ============================================================
# TÜM YOLLAR İÇİN TRAFİK YOĞUNLUĞU ANALİZİ
# YOLO11s + ByteTrack + Yol Bazlı Renkli Gösterim
# ============================================================

import cv2
import time
import subprocess
import numpy as np
import pandas as pd
import torch

from pathlib import Path
from ultralytics import YOLO
from IPython.display import Video, display
from google.colab import files


# ------------------------------------------------------------
# 1. AYARLAR
# ------------------------------------------------------------

# Daha önce video_yolu değişkeni tanımlandıysa onu kullanır.
# Tanımlanmadıysa aşağıdaki yolu kullanır.
video_yolu = globals().get("video_yolu", "/content/vid.mp4")

model_yolu = "yolo11s.pt"

gecici_video_yolu = "/content/tum_yollar_gecici.mp4"
sonuc_video_yolu = "/content/tum_yollar_trafik_yogunlugu.mp4"

# COCO araç sınıfları:
# 2 = car, 3 = motorcycle, 5 = bus, 7 = truck
arac_siniflari = [2, 3, 5, 7]

guven_esigi = 0.25
goruntu_boyutu = 960

if not Path(video_yolu).exists():
    raise FileNotFoundError(
        f"Video bulunamadı: {video_yolu}\n"
        "video_yolu değişkenini doğru dosya yoluyla değiştirin."
    )

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU bulunamadı. Colab'da Çalışma zamanı > "
        "Çalışma zamanı türünü değiştir > T4 GPU seçin."
    )

print("Kullanılan GPU:", torch.cuda.get_device_name(0))
print("Video:", video_yolu)


# ------------------------------------------------------------
# 2. VİDEO BİLGİLERİNİ OKUMA
# ------------------------------------------------------------

video = cv2.VideoCapture(video_yolu)

fps = video.get(cv2.CAP_PROP_FPS)
genislik = int(video.get(cv2.CAP_PROP_FRAME_WIDTH))
yukseklik = int(video.get(cv2.CAP_PROP_FRAME_HEIGHT))
toplam_kare = int(video.get(cv2.CAP_PROP_FRAME_COUNT))

video.release()

if fps <= 0 or genislik <= 0 or yukseklik <= 0:
    raise RuntimeError("Video bilgileri okunamadı.")

print("\nVideo çözünürlüğü:", genislik, "x", yukseklik)
print("FPS:", round(fps, 2))
print("Toplam kare:", toplam_kare)


# ------------------------------------------------------------
# 3. YOL BÖLGELERİNİ TANIMLAMA
# ------------------------------------------------------------

def nokta(x_orani, y_orani):
    """
    Oransal koordinatları piksel koordinatına dönüştürür.
    Böylece farklı çözünürlüklerde aynı yol geometrisi korunur.
    """
    return [
        int(x_orani * genislik),
        int(y_orani * yukseklik)
    ]


yol_bolgeleri = {
    "SOL ANA YOL": np.array([
        nokta(0.00, 0.43),
        nokta(0.52, 0.43),
        nokta(0.52, 0.66),
        nokta(0.00, 0.66)
    ], dtype=np.int32),

    "SAG ANA YOL": np.array([
        nokta(0.52, 0.43),
        nokta(1.00, 0.43),
        nokta(1.00, 0.66),
        nokta(0.52, 0.66)
    ], dtype=np.int32),

    "UST YOL": np.array([
        nokta(0.55, 0.02),
        nokta(0.76, 0.02),
        nokta(0.59, 0.46),
        nokta(0.49, 0.46)
    ], dtype=np.int32),

    "ALT YOL": np.array([
        # Alt yol görüntüde sola doğru ilerlediği için
        # çokgen de aşağıya indikçe sola kaydırılmıştır.
        nokta(0.40, 0.56),
        nokta(0.53, 0.56),
        nokta(0.48, 1.00),
        nokta(0.12, 1.00)
    ], dtype=np.int32)
}


# Her yol için ayrı maske oluştur
yol_maskeleri = {}
yol_alanlari = {}

for yol_adi, cokgen in yol_bolgeleri.items():

    maske = np.zeros(
        (yukseklik, genislik),
        dtype=np.uint8
    )

    cv2.fillPoly(
        maske,
        [cokgen],
        255
    )

    yol_maskeleri[yol_adi] = maske
    yol_alanlari[yol_adi] = cv2.countNonZero(maske)


# ------------------------------------------------------------
# 4. YOLO11s + BYTETRACK İLE ARAÇ TAKİBİ
# ------------------------------------------------------------

print("\nYOLO11s ve ByteTrack analizi başladı...")

model = YOLO(model_yolu)

gpu_baslangic = time.perf_counter()

takip_sonuclari = model.track(
    source=video_yolu,
    tracker="bytetrack.yaml",
    persist=True,
    classes=arac_siniflari,
    conf=guven_esigi,
    imgsz=goruntu_boyutu,
    device=0,
    quantize=16,
    stream=True,
    verbose=False
)

kare_kayitlari = []
doluluk_kayitlari = []
benzersiz_idler = set()


for kare_numarasi, sonuc in enumerate(takip_sonuclari):

    # Araç kutuları
    if sonuc.boxes is not None and len(sonuc.boxes) > 0:

        kutular = sonuc.boxes.xyxy.cpu().numpy().astype(int)
        siniflar = sonuc.boxes.cls.cpu().numpy().astype(int)

        if sonuc.boxes.id is not None:
            takip_idleri = sonuc.boxes.id.cpu().numpy().astype(int)
        else:
            takip_idleri = np.full(len(kutular), -1)

    else:
        kutular = np.empty((0, 4), dtype=int)
        siniflar = np.empty(0, dtype=int)
        takip_idleri = np.empty(0, dtype=int)

    for takip_id in takip_idleri:
        if takip_id >= 0:
            benzersiz_idler.add(int(takip_id))

    # Bu karedeki tüm araç kutularını maskele
    arac_maskesi = np.zeros(
        (yukseklik, genislik),
        dtype=np.uint8
    )

    for kutu in kutular:

        x1, y1, x2, y2 = kutu

        x1 = np.clip(x1, 0, genislik - 1)
        x2 = np.clip(x2, 0, genislik - 1)
        y1 = np.clip(y1, 0, yukseklik - 1)
        y2 = np.clip(y2, 0, yukseklik - 1)

        cv2.rectangle(
            arac_maskesi,
            (x1, y1),
            (x2, y2),
            255,
            -1
        )

    kare_doluluklari = {}

    # Her yolun araçlarla kaplanan alanını ayrı hesapla
    for yol_adi, yol_maskesi in yol_maskeleri.items():

        yol_uzerindeki_arac_alani = cv2.bitwise_and(
            arac_maskesi,
            yol_maskesi
        )

        kaplanan_alan = cv2.countNonZero(
            yol_uzerindeki_arac_alani
        )

        doluluk_yuzdesi = (
            kaplanan_alan /
            yol_alanlari[yol_adi]
        ) * 100

        kare_doluluklari[yol_adi] = doluluk_yuzdesi

    kare_kayitlari.append({
        "kare": kare_numarasi,
        "kutular": kutular,
        "siniflar": siniflar,
        "takip_idleri": takip_idleri,
        "aktif_arac": len(kutular)
    })

    doluluk_kayitlari.append(kare_doluluklari)


gpu_suresi = time.perf_counter() - gpu_baslangic

print("Araç analizi tamamlandı.")
print("GPU analiz süresi:", round(gpu_suresi, 2), "saniye")
print("Benzersiz takip ID sayısı:", len(benzersiz_idler))


# ------------------------------------------------------------
# 5. DOLULUK VERİ TABLOSUNU OLUŞTURMA
# ------------------------------------------------------------

grafik_verisi = pd.DataFrame(doluluk_kayitlari)

grafik_verisi.insert(
    0,
    "zaman_saniye",
    np.arange(len(grafik_verisi)) / fps
)

# Yaklaşık bir saniyelik hareketli ortalama
pencere_boyutu = max(1, int(round(fps)))

for yol_adi in yol_bolgeleri:

    grafik_verisi[yol_adi + "_ORT"] = (
        grafik_verisi[yol_adi]
        .rolling(
            window=pencere_boyutu,
            center=True,
            min_periods=1
        )
        .mean()
    )


# ------------------------------------------------------------
# 6. ÖNCEKİ ANALİZDEN ALINAN ORTAK YOĞUNLUK EŞİKLERİ
# ------------------------------------------------------------

# Önceki ana yol analizinde hesaplanan eşikler:
# %3.15 ve altı       -> DÜŞÜK
# %3.15 - %3.83 arası -> ORTA
# %3.83 üzeri         -> YÜKSEK
ortak_alt_esik = 3.15
ortak_ust_esik = 3.83

yol_esikleri = {
    yol_adi: {
        "alt": ortak_alt_esik,
        "ust": ortak_ust_esik
    }
    for yol_adi in yol_bolgeleri
}

print("\nKullanılan ortak yoğunluk eşikleri:")
print(f"DÜŞÜK : doluluk <= %{ortak_alt_esik:.2f}")
print(
    f"ORTA  : %{ortak_alt_esik:.2f} ile "
    f"%{ortak_ust_esik:.2f} arası"
)
print(f"YÜKSEK: doluluk > %{ortak_ust_esik:.2f}")


def yogunluk_sinifi(deger, alt_esik, ust_esik):

    # Yol doluluk değeri bütün video boyunca neredeyse
    # değişmiyorsa yanlış sınıflandırmayı engeller.
    if abs(ust_esik - alt_esik) < 0.001:
        return "DUSUK"

    if deger <= alt_esik:
        return "DUSUK"

    elif deger <= ust_esik:
        return "ORTA"

    else:
        return "YUKSEK"


# OpenCV renkleri BGR sırasındadır
renk_haritasi = {
    "DUSUK":  (0, 255, 0),     # Yeşil
    "ORTA":   (0, 255, 255),   # Sarı
    "YUKSEK": (0, 0, 255)      # Kırmızı
}


# ------------------------------------------------------------
# 7. RENKLİ SONUÇ VİDEOSUNU CPU İLE OLUŞTURMA
# ------------------------------------------------------------

print("\nRenkli sonuç videosu oluşturuluyor...")

cpu_baslangic = time.perf_counter()

giris_video = cv2.VideoCapture(video_yolu)

video_yazici = cv2.VideoWriter(
    gecici_video_yolu,
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (genislik, yukseklik)
)

kare_numarasi = 0

while True:

    basarili, kare = giris_video.read()

    if not basarili:
        break

    if kare_numarasi >= len(kare_kayitlari):
        break

    kayit = kare_kayitlari[kare_numarasi]
    renkli_katman = kare.copy()

    # Her yolu kendi yoğunluğuna göre renklendir
    for yol_adi, cokgen in yol_bolgeleri.items():

        doluluk = grafik_verisi.loc[
            kare_numarasi,
            yol_adi + "_ORT"
        ]

        alt_esik = yol_esikleri[yol_adi]["alt"]
        ust_esik = yol_esikleri[yol_adi]["ust"]

        sinif = yogunluk_sinifi(
            doluluk,
            alt_esik,
            ust_esik
        )

        renk = renk_haritasi[sinif]

        cv2.fillPoly(
            renkli_katman,
            [cokgen],
            renk
        )

    # Saydam renkli katmanı gerçek kareyle birleştir
    kare = cv2.addWeighted(
        renkli_katman,
        0.28,
        kare,
        0.72,
        0
    )

    # Yol sınırlarını ve bilgilerini çiz
    yazi_y = 35

    for yol_adi, cokgen in yol_bolgeleri.items():

        doluluk = grafik_verisi.loc[
            kare_numarasi,
            yol_adi + "_ORT"
        ]

        alt_esik = yol_esikleri[yol_adi]["alt"]
        ust_esik = yol_esikleri[yol_adi]["ust"]

        sinif = yogunluk_sinifi(
            doluluk,
            alt_esik,
            ust_esik
        )

        renk = renk_haritasi[sinif]

        cv2.polylines(
            kare,
            [cokgen],
            True,
            renk,
            3
        )

        bilgi = (
            f"{yol_adi}: {sinif} - %{doluluk:.2f}"
        )

        # Yazının görünmesi için siyah arka plan
        cv2.rectangle(
            kare,
            (10, yazi_y - 24),
            (470, yazi_y + 7),
            (0, 0, 0),
            -1
        )

        cv2.putText(
            kare,
            bilgi,
            (18, yazi_y),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.65,
            renk,
            2
        )

        yazi_y += 38

    # Araç kutularını ve ByteTrack numaralarını çiz
    for kutu, takip_id, sinif_no in zip(
        kayit["kutular"],
        kayit["takip_idleri"],
        kayit["siniflar"]
    ):

        x1, y1, x2, y2 = map(int, kutu)

        cv2.rectangle(
            kare,
            (x1, y1),
            (x2, y2),
            (255, 255, 255),
            2
        )

        if takip_id >= 0:
            arac_yazisi = f"ID:{takip_id}"
        else:
            arac_yazisi = "ARAC"

        cv2.putText(
            kare,
            arac_yazisi,
            (x1, max(20, y1 - 7)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.55,
            (255, 255, 255),
            2
        )

    # Genel bilgi
    cv2.putText(
        kare,
        f"Aktif arac: {kayit['aktif_arac']}",
        (genislik - 260, 35),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (255, 255, 255),
        2
    )

    video_yazici.write(kare)

    kare_numarasi += 1


giris_video.release()
video_yazici.release()

cpu_video_suresi = time.perf_counter() - cpu_baslangic

print(
    "CPU ile renkli video oluşturma süresi:",
    round(cpu_video_suresi, 2),
    "saniye"
)


# ------------------------------------------------------------
# 8. VİDEOYU H.264 FORMATINA DÖNÜŞTÜRME
# ------------------------------------------------------------

ffmpeg_baslangic = time.perf_counter()

subprocess.run(
    [
        "ffmpeg",
        "-y",
        "-i", gecici_video_yolu,
        "-vcodec", "libx264",
        "-pix_fmt", "yuv420p",
        "-movflags", "+faststart",
        sonuc_video_yolu
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    check=True
)

ffmpeg_suresi = time.perf_counter() - ffmpeg_baslangic


# ------------------------------------------------------------
# 9. SÜRE VE SONUÇ BİLGİLERİ
# ------------------------------------------------------------

toplam_sure = (
    gpu_suresi +
    cpu_video_suresi +
    ffmpeg_suresi
)

print("\n========== İŞLEM SÜRELERİ ==========")
print("YOLO + ByteTrack analiz süresi :", round(gpu_suresi, 2), "sn")
print("CPU görüntü oluşturma süresi   :", round(cpu_video_suresi, 2), "sn")
print("FFmpeg dönüştürme süresi       :", round(ffmpeg_suresi, 2), "sn")
print("Toplam işlem süresi            :", round(toplam_sure, 2), "sn")
print("Sonuç videosu                  :", sonuc_video_yolu)


# ------------------------------------------------------------
# 10. SONUÇ VİDEOSUNU GÖSTERME
# ------------------------------------------------------------

display(
    Video(
        sonuc_video_yolu,
        embed=True,
        width=900
    )
)

# **BU AŞAMADA GPU KULLANILMAYACAKTIR. SADECE CPU İLE İŞLEM YAPILACAKTIR.**

# ***Hata Giderme Aşaması***

# ***# Yol Bölgelerinin Düzeltilmesi***

Aşama 1 — Kaynak Trafik Videosunun İnternetten Colab’a İndirilmesi

In [ ]:
import subprocess
from pathlib import Path


video_url = (
    "https://raw.githubusercontent.com/"
    "antonmilev/TrafficDetection/main/vid.mp4"
)

video_yolu = "/content/trafik_akisi.mp4"


subprocess.run(
    [
        "wget",
        "-O",
        video_yolu,
        video_url
    ],
    check=True
)


if not Path(video_yolu).exists():
    raise FileNotFoundError(
        "Video indirilemedi."
    )


print("Video başarıyla indirildi.")
print("Video yolu:", video_yolu)

Bu aşamada indirilen videonun 2. saniyesinden bir kare alınır. Görüntü özgün 1920 × 1080 çözünürlükte kaydedilir ve Paint’te işaretlemek üzere bilgisayara indirilir.

In [ ]:
import cv2

from google.colab.patches import cv2_imshow
from google.colab import files


video = cv2.VideoCapture(video_yolu)

# Videonun 2. saniyesine git
video.set(
    cv2.CAP_PROP_POS_MSEC,
    2000
)

basarili, yol_secim_karesi = video.read()

video.release()


if not basarili:
    raise RuntimeError(
        "Videodan görüntü alınamadı."
    )


resim_yolu = "/content/yol_secim_karesi.jpg"

kaydedildi = cv2.imwrite(
    resim_yolu,
    yol_secim_karesi
)


if not kaydedildi:
    raise RuntimeError(
        "Görüntü kaydedilemedi."
    )


print(
    "Görüntü boyutu:",
    yol_secim_karesi.shape[1],
    "x",
    yol_secim_karesi.shape[0]
)

cv2_imshow(yol_secim_karesi)

files.download(resim_yolu)

## **Aşama 3 — Alt Yol Sınırının Paint ile İşaretlenmesi**

indirilen yol_secim_karesi.jpg görüntüsünü Paint’te açın.

Uygulanacak işlem:

1-Çizgi aracını seçin.

2-Siyah veya kırmızı, belirgin bir renk kullanın.

3-Yalnızca alt yolun görünen asfalt sınırlarını çevreleyin.

4-Ağaç, kaldırım ve çimenlik alanları dışarıda bırakın.

5-Çizgileri birleştirerek kapalı bir çokgen oluşturun.

Görüntüyü yeniden boyutlandırmayın ve kırpmayın.

alt_yol_isaretli.png adıyla farklı kaydedin.

İçini boyamayın; yalnızca dış sınırı çizmeniz yeterli. İşaretlenmiş görüntüyü buraya yükleyin. Çizgileri 1920 * 1080 koordinatlarına dönüştürerek alt yol maskesini oluşturacağız.

# **Aşama 4 — Paint Çizgisinden Yol Koordinatlarının Çıkarılması**

In [ ]:
import cv2
import numpy as np

from google.colab import files
from google.colab.patches import cv2_imshow


# -------------------------------------------------
# 1. Paint'te işaretlenen görüntüyü yükle
# -------------------------------------------------

yuklenen_dosyalar = files.upload()

isaretli_dosya_adi = next(
    iter(yuklenen_dosyalar.keys())
)

isaretli_gorsel = cv2.imread(
    isaretli_dosya_adi
)


if isaretli_gorsel is None:
    raise RuntimeError(
        "İşaretli görüntü okunamadı."
    )


# -------------------------------------------------
# 2. Orijinal görüntüyü oku
# -------------------------------------------------

orijinal_resim_yolu = (
    "/content/yol_secim_karesi.jpg"
)

orijinal_gorsel = cv2.imread(
    orijinal_resim_yolu
)


if orijinal_gorsel is None:
    raise RuntimeError(
        "Orijinal yol seçim görüntüsü bulunamadı."
    )


if (
    orijinal_gorsel.shape
    != isaretli_gorsel.shape
):
    raise RuntimeError(
        "İki görüntünün boyutları aynı değil. "
        "Paint görüntüsünü kırpmayın veya "
        "yeniden boyutlandırmayın."
    )


# -------------------------------------------------
# 3. Paint'te eklenen çizgiyi bul
# -------------------------------------------------

fark = cv2.absdiff(
    orijinal_gorsel,
    isaretli_gorsel
)

fark_miktari = np.max(
    fark,
    axis=2
)

cizgi_maskesi = np.where(
    fark_miktari > 60,
    255,
    0
).astype(np.uint8)


# Çizgideki küçük boşlukları birleştir
cekirdek = np.ones(
    (7, 7),
    dtype=np.uint8
)

cizgi_maskesi = cv2.morphologyEx(
    cizgi_maskesi,
    cv2.MORPH_CLOSE,
    cekirdek,
    iterations=2
)


# -------------------------------------------------
# 4. En büyük kapalı çizgiyi bul
# -------------------------------------------------

konturlar, _ = cv2.findContours(
    cizgi_maskesi,
    cv2.RETR_EXTERNAL,
    cv2.CHAIN_APPROX_SIMPLE
)


if not konturlar:
    raise RuntimeError(
        "Paint çizgisi bulunamadı."
    )


en_buyuk_kontur = max(
    konturlar,
    key=cv2.contourArea
)

dis_bukey = cv2.convexHull(
    en_buyuk_kontur
)

cevre = cv2.arcLength(
    dis_bukey,
    True
)


# Dört köşeli çokgeni bul
alt_yol_noktalari = None

for hassasiyet in np.linspace(
    0.005,
    0.10,
    50
):

    yaklasik_cokgen = cv2.approxPolyDP(
        dis_bukey,
        hassasiyet * cevre,
        True
    )

    if len(yaklasik_cokgen) == 4:

        alt_yol_noktalari = (
            yaklasik_cokgen
            .reshape(-1, 2)
            .astype(np.int32)
        )

        break


if alt_yol_noktalari is None:
    raise RuntimeError(
        "Çizgiden dört köşe çıkarılamadı. "
        "Paint çizgisini daha kalın ve kapalı çizin."
    )


# -------------------------------------------------
# 5. Bulunan alanı ön izle
# -------------------------------------------------

katman = orijinal_gorsel.copy()

cv2.fillPoly(
    katman,
    [alt_yol_noktalari],
    (0, 255, 0)
)

onizleme = cv2.addWeighted(
    katman,
    0.30,
    orijinal_gorsel,
    0.70,
    0
)

cv2.polylines(
    onizleme,
    [alt_yol_noktalari],
    True,
    (0, 255, 0),
    5
)


# Ekrana sığdır
kucuk_onizleme = cv2.resize(
    onizleme,
    (960, 540)
)

cv2_imshow(kucuk_onizleme)


# -------------------------------------------------
# 6. Koordinatları yazdır
# -------------------------------------------------

yukseklik, genislik = (
    orijinal_gorsel.shape[:2]
)

print("\nPiksel koordinatları:")
print(alt_yol_noktalari)

print("\nOransal koordinatlar:")

for x, y in alt_yol_noktalari:

    print(
        f"nokta({x / genislik:.4f}, "
        f"{y / yukseklik:.4f}),"
    )

# **Aşama 5 — Alt Yol Maskesinin Video Üzerine Yerleştirilmesi**

In [ ]:
import cv2
import subprocess

from IPython.display import Video, display


gecici_video = (
    "/content/alt_yol_kontrol_gecici.mp4"
)

sonuc_video = (
    "/content/alt_yol_kontrol.mp4"
)


giris_video = cv2.VideoCapture(
    video_yolu
)

fps = giris_video.get(
    cv2.CAP_PROP_FPS
)

genislik = int(
    giris_video.get(
        cv2.CAP_PROP_FRAME_WIDTH
    )
)

yukseklik = int(
    giris_video.get(
        cv2.CAP_PROP_FRAME_HEIGHT
    )
)


# Görüntü ve video aynı boyutta olmalı
if (
    genislik != 1920
    or yukseklik != 1080
):
    raise RuntimeError(
        "Video çözünürlüğü 1920 × 1080 değil."
    )


video_yazici = cv2.VideoWriter(
    gecici_video,
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (genislik, yukseklik)
)


while True:

    basarili, kare = giris_video.read()

    if not basarili:
        break

    katman = kare.copy()

    # Alt yolun içini yeşil boya
    cv2.fillPoly(
        katman,
        [alt_yol_noktalari],
        (0, 255, 0)
    )

    # Saydam katmanı görüntüyle birleştir
    kare = cv2.addWeighted(
        katman,
        0.30,
        kare,
        0.70,
        0
    )

    # Alt yolun dış sınırını çiz
    cv2.polylines(
        kare,
        [alt_yol_noktalari],
        True,
        (0, 255, 0),
        5
    )

    cv2.putText(
        kare,
        "ALT YOL",
        (640, 615),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (0, 255, 0),
        3
    )

    video_yazici.write(kare)


giris_video.release()
video_yazici.release()


# Tarayıcı uyumlu H.264 videosu oluştur
subprocess.run(
    [
        "ffmpeg",
        "-y",
        "-loglevel",
        "error",
        "-i",
        gecici_video,
        "-vcodec",
        "libx264",
        "-pix_fmt",
        "yuv420p",
        sonuc_video
    ],
    check=True
)


display(
    Video(
        sonuc_video,
        embed=True,
        width=900
    )
)

# **Aşama 6 — Sağ Ana Yolun Paint’te İşaretlenmesi**

İşaretsiz yol_secim_karesi.jpg görüntüsünü Paint’te yeniden açın.

Yalnızca sağ taraftaki yatay yolun asfalt bölümünü kapalı bir çokgenle çevreleyin.

Yolun üst ve alt sınırlarını takip edin.

Kaldırım, çimen, ağaç ve refüjleri dışarıda bırakın.

Orta kavşak bölgesinde sınırı, sağ yolun başladığı noktada bitirin.

Görüntüyü kırpmayın veya yeniden boyutlandırmayın.

sag_yol_isaretli.png adıyla kaydedin.

In [ ]:
import cv2
import numpy as np

from google.colab import files
from google.colab.patches import cv2_imshow


# İşaretli sağ yol görselini yükle
yuklenen = files.upload()

isaretli_dosya = next(
    iter(yuklenen.keys())
)


orijinal_gorsel = cv2.imread(
    "/content/yol_secim_karesi.jpg"
)

isaretli_gorsel = cv2.imread(
    isaretli_dosya
)


if orijinal_gorsel is None:
    raise RuntimeError(
        "Orijinal görüntü bulunamadı."
    )

if isaretli_gorsel is None:
    raise RuntimeError(
        "İşaretli görüntü okunamadı."
    )

if (
    orijinal_gorsel.shape
    != isaretli_gorsel.shape
):
    raise RuntimeError(
        "Görüntü boyutları aynı değil. "
        "Paint görüntüsünü kırpmayın."
    )


# Paint çizgisini orijinalden ayır
fark = cv2.absdiff(
    orijinal_gorsel,
    isaretli_gorsel
)

fark_miktari = np.max(
    fark,
    axis=2
)

cizgi_maskesi = np.where(
    fark_miktari > 60,
    255,
    0
).astype(np.uint8)


# Çizgideki küçük boşlukları kapat
cekirdek = np.ones(
    (7, 7),
    dtype=np.uint8
)

cizgi_maskesi = cv2.morphologyEx(
    cizgi_maskesi,
    cv2.MORPH_CLOSE,
    cekirdek,
    iterations=2
)


# En büyük kapalı çizgiyi bul
konturlar, _ = cv2.findContours(
    cizgi_maskesi,
    cv2.RETR_EXTERNAL,
    cv2.CHAIN_APPROX_SIMPLE
)

if not konturlar:
    raise RuntimeError(
        "Paint çizgisi bulunamadı."
    )


en_buyuk_kontur = max(
    konturlar,
    key=cv2.contourArea
)

dis_bukey = cv2.convexHull(
    en_buyuk_kontur
)

cevre = cv2.arcLength(
    dis_bukey,
    True
)


# Dört köşeyi çıkar
sag_yol_noktalari = None

for hassasiyet in np.linspace(
    0.005,
    0.10,
    50
):

    cokgen = cv2.approxPolyDP(
        dis_bukey,
        hassasiyet * cevre,
        True
    )

    if len(cokgen) == 4:

        sag_yol_noktalari = (
            cokgen
            .reshape(-1, 2)
            .astype(np.int32)
        )

        break


if sag_yol_noktalari is None:
    raise RuntimeError(
        "Dört köşe çıkarılamadı. "
        "Çizgiyi kapalı ve belirgin çizin."
    )


# Sağ yol ön izlemesi
katman = orijinal_gorsel.copy()

cv2.fillPoly(
    katman,
    [sag_yol_noktalari],
    (0, 255, 0)
)

onizleme = cv2.addWeighted(
    katman,
    0.30,
    orijinal_gorsel,
    0.70,
    0
)

cv2.polylines(
    onizleme,
    [sag_yol_noktalari],
    True,
    (0, 255, 0),
    5
)

kucuk_onizleme = cv2.resize(
    onizleme,
    (960, 540)
)

cv2_imshow(kucuk_onizleme)


# Koordinatları yazdır
yukseklik, genislik = (
    orijinal_gorsel.shape[:2]
)

print("\nSağ yol piksel koordinatları:")
print(sag_yol_noktalari)

print("\nSağ yol oransal koordinatları:")

for x, y in sag_yol_noktalari:

    print(
        f"nokta({x / genislik:.4f}, "
        f"{y / yukseklik:.4f}),"
    )

# **Aşama 7 — Üst Yolun Paint ile İşaretlenmesi**

İşaretsiz yol_secim_karesi.jpg dosyasını Paint’te açın.

Yalnızca üst taraftan kavşağa gelen yolun asfaltını çevreleyin.

Park alanlarını, kaldırımı, ağaçları ve binaları dışarıda bırakın.

Kapalı bir çokgen çizin.

Görüntüyü kırpmadan ust_yol_isaretli.png adıyla kaydedin.

# **Aşama 7 — Paint Çizgisinden Üst Yol Koordinatlarının Çıkarılması**

In [ ]:
import cv2
import numpy as np

from google.colab import files
from google.colab.patches import cv2_imshow


# -------------------------------------------------
# 1. İşaretlenmiş üst yol görselini yükle
# -------------------------------------------------

yuklenen = files.upload()

isaretli_dosya = next(
    iter(yuklenen.keys())
)

isaretli_gorsel = cv2.imread(
    isaretli_dosya
)


if isaretli_gorsel is None:
    raise RuntimeError(
        "İşaretli görüntü okunamadı."
    )


# -------------------------------------------------
# 2. Orijinal görüntüyü oku
# -------------------------------------------------

orijinal_resim_yolu = (
    "/content/yol_secim_karesi.jpg"
)

orijinal_gorsel = cv2.imread(
    orijinal_resim_yolu
)


if orijinal_gorsel is None:
    raise RuntimeError(
        "Orijinal görüntü bulunamadı."
    )


if (
    orijinal_gorsel.shape
    != isaretli_gorsel.shape
):
    raise RuntimeError(
        "Görüntü boyutları aynı değil. "
        "Paint görüntüsünü kırpmayın "
        "veya yeniden boyutlandırmayın."
    )


# -------------------------------------------------
# 3. Paint çizgisini bul
# -------------------------------------------------

fark = cv2.absdiff(
    orijinal_gorsel,
    isaretli_gorsel
)

fark_miktari = np.max(
    fark,
    axis=2
)

cizgi_maskesi = np.where(
    fark_miktari > 60,
    255,
    0
).astype(np.uint8)


# Çizgideki küçük boşlukları birleştir
cekirdek = np.ones(
    (7, 7),
    dtype=np.uint8
)

cizgi_maskesi = cv2.morphologyEx(
    cizgi_maskesi,
    cv2.MORPH_CLOSE,
    cekirdek,
    iterations=2
)


# -------------------------------------------------
# 4. En büyük kapalı çizgiyi bul
# -------------------------------------------------

konturlar, _ = cv2.findContours(
    cizgi_maskesi,
    cv2.RETR_EXTERNAL,
    cv2.CHAIN_APPROX_SIMPLE
)


if not konturlar:
    raise RuntimeError(
        "Paint çizgisi bulunamadı."
    )


en_buyuk_kontur = max(
    konturlar,
    key=cv2.contourArea
)

dis_bukey = cv2.convexHull(
    en_buyuk_kontur
)

cevre = cv2.arcLength(
    dis_bukey,
    True
)


# -------------------------------------------------
# 5. Üst yolun dört köşesini çıkar
# -------------------------------------------------

ust_yol_noktalari = None

for hassasiyet in np.linspace(
    0.005,
    0.10,
    50
):

    cokgen = cv2.approxPolyDP(
        dis_bukey,
        hassasiyet * cevre,
        True
    )

    if len(cokgen) == 4:

        ust_yol_noktalari = (
            cokgen
            .reshape(-1, 2)
            .astype(np.int32)
        )

        break


if ust_yol_noktalari is None:
    raise RuntimeError(
        "Dört köşe çıkarılamadı. "
        "Paint çizgisini kapalı ve "
        "daha belirgin çizin."
    )


# -------------------------------------------------
# 6. Bulunan üst yol alanını göster
# -------------------------------------------------

katman = orijinal_gorsel.copy()

cv2.fillPoly(
    katman,
    [ust_yol_noktalari],
    (0, 255, 0)
)

onizleme = cv2.addWeighted(
    katman,
    0.30,
    orijinal_gorsel,
    0.70,
    0
)

cv2.polylines(
    onizleme,
    [ust_yol_noktalari],
    True,
    (0, 255, 0),
    5
)


# Yalnızca ekrandaki ön izlemeyi küçült
kucuk_onizleme = cv2.resize(
    onizleme,
    (960, 540)
)

cv2_imshow(kucuk_onizleme)


# -------------------------------------------------
# 7. Koordinatları yazdır
# -------------------------------------------------

yukseklik, genislik = (
    orijinal_gorsel.shape[:2]
)

print("\nÜst yol piksel koordinatları:")
print(ust_yol_noktalari)

print("\nÜst yol oransal koordinatları:")

for x, y in ust_yol_noktalari:

    print(
        f"nokta({x / genislik:.4f}, "
        f"{y / yukseklik:.4f}),"
    )

# **Aşama 8 — Paint Çizgisinden Sol Yol Koordinatlarının Çıkarılması**

In [ ]:
import cv2
import numpy as np

from google.colab import files
from google.colab.patches import cv2_imshow


# -------------------------------------------------
# 1. İşaretlenmiş sol yol görselini yükle
# -------------------------------------------------

yuklenen = files.upload()

isaretli_dosya = next(
    iter(yuklenen.keys())
)

isaretli_gorsel = cv2.imread(
    isaretli_dosya
)


if isaretli_gorsel is None:
    raise RuntimeError(
        "İşaretli görüntü okunamadı."
    )


# -------------------------------------------------
# 2. Orijinal görüntüyü oku
# -------------------------------------------------

orijinal_resim_yolu = (
    "/content/yol_secim_karesi.jpg"
)

orijinal_gorsel = cv2.imread(
    orijinal_resim_yolu
)


if orijinal_gorsel is None:
    raise RuntimeError(
        "Orijinal görüntü bulunamadı."
    )


if (
    orijinal_gorsel.shape
    != isaretli_gorsel.shape
):
    raise RuntimeError(
        "Görüntü boyutları aynı değil. "
        "Paint görüntüsünü kırpmayın "
        "veya yeniden boyutlandırmayın."
    )


# -------------------------------------------------
# 3. Paint çizgisini bul
# -------------------------------------------------

fark = cv2.absdiff(
    orijinal_gorsel,
    isaretli_gorsel
)

fark_miktari = np.max(
    fark,
    axis=2
)

cizgi_maskesi = np.where(
    fark_miktari > 60,
    255,
    0
).astype(np.uint8)


# Çizgideki küçük boşlukları birleştir
cekirdek = np.ones(
    (7, 7),
    dtype=np.uint8
)

cizgi_maskesi = cv2.morphologyEx(
    cizgi_maskesi,
    cv2.MORPH_CLOSE,
    cekirdek,
    iterations=2
)


# -------------------------------------------------
# 4. En büyük kapalı çizgiyi bul
# -------------------------------------------------

konturlar, _ = cv2.findContours(
    cizgi_maskesi,
    cv2.RETR_EXTERNAL,
    cv2.CHAIN_APPROX_SIMPLE
)


if not konturlar:
    raise RuntimeError(
        "Paint çizgisi bulunamadı."
    )


en_buyuk_kontur = max(
    konturlar,
    key=cv2.contourArea
)

dis_bukey = cv2.convexHull(
    en_buyuk_kontur
)

cevre = cv2.arcLength(
    dis_bukey,
    True
)


# -------------------------------------------------
# 5. Sol yolun dört köşesini çıkar
# -------------------------------------------------

sol_yol_noktalari = None

for hassasiyet in np.linspace(
    0.005,
    0.10,
    50
):

    cokgen = cv2.approxPolyDP(
        dis_bukey,
        hassasiyet * cevre,
        True
    )

    if len(cokgen) == 4:

        sol_yol_noktalari = (
            cokgen
            .reshape(-1, 2)
            .astype(np.int32)
        )

        break


if sol_yol_noktalari is None:
    raise RuntimeError(
        "Dört köşe çıkarılamadı. "
        "Paint çizgisini kapalı ve "
        "daha belirgin çizin."
    )


# -------------------------------------------------
# 6. Bulunan sol yol alanını göster
# -------------------------------------------------

katman = orijinal_gorsel.copy()

cv2.fillPoly(
    katman,
    [sol_yol_noktalari],
    (0, 255, 0)
)

onizleme = cv2.addWeighted(
    katman,
    0.30,
    orijinal_gorsel,
    0.70,
    0
)

cv2.polylines(
    onizleme,
    [sol_yol_noktalari],
    True,
    (0, 255, 0),
    5
)


# Yalnızca ön izlemeyi küçült
kucuk_onizleme = cv2.resize(
    onizleme,
    (960, 540)
)

cv2_imshow(kucuk_onizleme)


# -------------------------------------------------
# 7. Koordinatları yazdır
# -------------------------------------------------

yukseklik, genislik = (
    orijinal_gorsel.shape[:2]
)

print("\nSol yol piksel koordinatları:")
print(sol_yol_noktalari)

print("\nSol yol oransal koordinatları:")

for x, y in sol_yol_noktalari:

    print(
        f"nokta({x / genislik:.4f}, "
        f"{y / yukseklik:.4f}),"
    )

# **Aşama 9 — Dört Yol Bölgesinin Birlikte Kontrol Edilmesi**

In [ ]:
import cv2
import numpy as np

from google.colab.patches import cv2_imshow


# Dört yol bölgesini birleştir
yol_bolgeleri = {
    "SOL YOL": sol_yol_noktalari,
    "SAG YOL": sag_yol_noktalari,
    "UST YOL": ust_yol_noktalari,
    "ALT YOL": alt_yol_noktalari
}


# OpenCV renkleri BGR sırasındadır
yol_renkleri = {
    "SOL YOL": (255, 0, 0),      # Mavi
    "SAG YOL": (0, 255, 0),      # Yeşil
    "UST YOL": (0, 0, 255),      # Kırmızı
    "ALT YOL": (0, 255, 255)     # Sarı
}


orijinal_gorsel = cv2.imread(
    "/content/yol_secim_karesi.jpg"
)

if orijinal_gorsel is None:
    raise RuntimeError(
        "Orijinal görüntü bulunamadı."
    )


renkli_katman = orijinal_gorsel.copy()


# Yol bölgelerinin içini boya
for yol_adi, yol_noktalari in yol_bolgeleri.items():

    cv2.fillPoly(
        renkli_katman,
        [yol_noktalari],
        yol_renkleri[yol_adi]
    )


# Saydam katmanı görüntüyle birleştir
onizleme = cv2.addWeighted(
    renkli_katman,
    0.30,
    orijinal_gorsel,
    0.70,
    0
)


# Sınırları ve yol adlarını çiz
for yol_adi, yol_noktalari in yol_bolgeleri.items():

    renk = yol_renkleri[yol_adi]

    cv2.polylines(
        onizleme,
        [yol_noktalari],
        True,
        renk,
        5
    )

    merkez = np.mean(
        yol_noktalari,
        axis=0
    ).astype(int)

    cv2.putText(
        onizleme,
        yol_adi,
        tuple(merkez),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        renk,
        3
    )


# Ekrana sığdır
kucuk_onizleme = cv2.resize(
    onizleme,
    (960, 540)
)

cv2_imshow(kucuk_onizleme)

| Yol doluluk oranı             | Sınıf  | Renk       |
| ----------------------------- | ------ | ---------- |
| **%3 ve altı**                | Düşük  | 🟢 Yeşil   |
| **%3’ten büyük, %10 ve altı** | Orta   | 🟡 Sarı    |
| **%10’dan büyük**             | Yüksek | 🔴 Kırmızı |


In [ ]:
# ============================================================
# TÜM YOLLAR İÇİN TRAFİK YOĞUNLUĞU ANALİZİ
# YOLO11s + ByteTrack + Yol Bazlı Renkli Gösterim
# ============================================================

import cv2
import time
import subprocess
import numpy as np
import pandas as pd
import torch

from pathlib import Path
from ultralytics import YOLO
from IPython.display import Video, display
from google.colab import files


# ------------------------------------------------------------
# 1. AYARLAR
# ------------------------------------------------------------

# Daha önce video_yolu değişkeni tanımlandıysa onu kullanır.
# Tanımlanmadıysa aşağıdaki yolu kullanır.
video_yolu = globals().get("video_yolu", "/content/vid.mp4")

model_yolu = "yolo11s.pt"

gecici_video_yolu = "/content/tum_yollar_gecici.mp4"
sonuc_video_yolu = "/content/tum_yollar_trafik_yogunlugu.mp4"

# COCO araç sınıfları:
# 2 = car, 3 = motorcycle, 5 = bus, 7 = truck
arac_siniflari = [2, 3, 5, 7]

guven_esigi = 0.15
goruntu_boyutu = 960

if not Path(video_yolu).exists():
    raise FileNotFoundError(
        f"Video bulunamadı: {video_yolu}\n"
        "video_yolu değişkenini doğru dosya yoluyla değiştirin."
    )

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU bulunamadı. Colab'da Çalışma zamanı > "
        "Çalışma zamanı türünü değiştir > T4 GPU seçin."
    )

print("Kullanılan GPU:", torch.cuda.get_device_name(0))
print("Video:", video_yolu)


# ------------------------------------------------------------
# 2. VİDEO BİLGİLERİNİ OKUMA
# ------------------------------------------------------------

video = cv2.VideoCapture(video_yolu)

fps = video.get(cv2.CAP_PROP_FPS)
genislik = int(video.get(cv2.CAP_PROP_FRAME_WIDTH))
yukseklik = int(video.get(cv2.CAP_PROP_FRAME_HEIGHT))
toplam_kare = int(video.get(cv2.CAP_PROP_FRAME_COUNT))

video.release()

if fps <= 0 or genislik <= 0 or yukseklik <= 0:
    raise RuntimeError("Video bilgileri okunamadı.")

print("\nVideo çözünürlüğü:", genislik, "x", yukseklik)
print("FPS:", round(fps, 2))
print("Toplam kare:", toplam_kare)


# ------------------------------------------------------------
# 3. YOL BÖLGELERİNİ TANIMLAMA
# ------------------------------------------------------------

def nokta(x_orani, y_orani):
    """
    Oransal koordinatları piksel koordinatına dönüştürür.
    Böylece farklı çözünürlüklerde aynı yol geometrisi korunur.
    """
    return [
        int(x_orani * genislik),
        int(y_orani * yukseklik)
    ]


yol_bolgeleri = {
    # Paint çizgilerinden çıkarılan kesin yol sınırları
    "SOL YOL": np.array([
        nokta(0.3995, 0.4241),
        nokta(0.2984, 0.5787),
        nokta(0.0000, 0.5648),
        nokta(0.0000, 0.4315)
    ], dtype=np.int32),

    "SAG YOL": np.array([
        nokta(0.5151, 0.5880),
        nokta(0.5510, 0.4287),
        nokta(0.9958, 0.4694),
        nokta(0.9943, 0.6481)
    ], dtype=np.int32),

    "UST YOL": np.array([
        nokta(0.7359, 0.0000),
        nokta(0.5536, 0.4315),
        nokta(0.4224, 0.4287),
        nokta(0.6542, 0.0000)
    ], dtype=np.int32),

    "ALT YOL": np.array([
        nokta(0.4625, 0.5926),
        nokta(0.2953, 0.9991),
        nokta(0.1255, 0.9991),
        nokta(0.3292, 0.5824)
    ], dtype=np.int32)
}


# Her yol için ayrı maske oluştur
yol_maskeleri = {}
yol_alanlari = {}

for yol_adi, cokgen in yol_bolgeleri.items():

    maske = np.zeros(
        (yukseklik, genislik),
        dtype=np.uint8
    )

    cv2.fillPoly(
        maske,
        [cokgen],
        255
    )

    yol_maskeleri[yol_adi] = maske
    yol_alanlari[yol_adi] = cv2.countNonZero(maske)


# ------------------------------------------------------------
# 4. YOLO11s + BYTETRACK İLE ARAÇ TAKİBİ
# ------------------------------------------------------------

print("\nYOLO11s ve ByteTrack analizi başladı...")

model = YOLO(model_yolu)

gpu_baslangic = time.perf_counter()

takip_sonuclari = model.track(
    source=video_yolu,
    tracker="bytetrack.yaml",
    persist=True,
    classes=arac_siniflari,
    conf=guven_esigi,
    imgsz=goruntu_boyutu,
    device=0,
    quantize=16,
    stream=True,
    verbose=False
)

kare_kayitlari = []
doluluk_kayitlari = []
benzersiz_idler = set()


for kare_numarasi, sonuc in enumerate(takip_sonuclari):

    # Araç kutuları
    if sonuc.boxes is not None and len(sonuc.boxes) > 0:

        kutular = sonuc.boxes.xyxy.cpu().numpy().astype(int)
        siniflar = sonuc.boxes.cls.cpu().numpy().astype(int)

        if sonuc.boxes.id is not None:
            takip_idleri = sonuc.boxes.id.cpu().numpy().astype(int)
        else:
            takip_idleri = np.full(len(kutular), -1)

    else:
        kutular = np.empty((0, 4), dtype=int)
        siniflar = np.empty(0, dtype=int)
        takip_idleri = np.empty(0, dtype=int)

    for takip_id in takip_idleri:
        if takip_id >= 0:
            benzersiz_idler.add(int(takip_id))

    # Bu karedeki tüm araç kutularını maskele
    arac_maskesi = np.zeros(
        (yukseklik, genislik),
        dtype=np.uint8
    )

    for kutu in kutular:

        x1, y1, x2, y2 = kutu

        x1 = np.clip(x1, 0, genislik - 1)
        x2 = np.clip(x2, 0, genislik - 1)
        y1 = np.clip(y1, 0, yukseklik - 1)
        y2 = np.clip(y2, 0, yukseklik - 1)

        cv2.rectangle(
            arac_maskesi,
            (x1, y1),
            (x2, y2),
            255,
            -1
        )

    kare_doluluklari = {}

    # Her yolun araçlarla kaplanan alanını ayrı hesapla
    for yol_adi, yol_maskesi in yol_maskeleri.items():

        yol_uzerindeki_arac_alani = cv2.bitwise_and(
            arac_maskesi,
            yol_maskesi
        )

        kaplanan_alan = cv2.countNonZero(
            yol_uzerindeki_arac_alani
        )

        doluluk_yuzdesi = (
            kaplanan_alan /
            yol_alanlari[yol_adi]
        ) * 100

        kare_doluluklari[yol_adi] = doluluk_yuzdesi

    kare_kayitlari.append({
        "kare": kare_numarasi,
        "kutular": kutular,
        "siniflar": siniflar,
        "takip_idleri": takip_idleri,
        "aktif_arac": len(kutular)
    })

    doluluk_kayitlari.append(kare_doluluklari)


gpu_suresi = time.perf_counter() - gpu_baslangic

print("Araç analizi tamamlandı.")
print("GPU analiz süresi:", round(gpu_suresi, 2), "saniye")
print("Benzersiz takip ID sayısı:", len(benzersiz_idler))


# ------------------------------------------------------------
# 5. DOLULUK VERİ TABLOSUNU OLUŞTURMA
# ------------------------------------------------------------

grafik_verisi = pd.DataFrame(doluluk_kayitlari)

grafik_verisi.insert(
    0,
    "zaman_saniye",
    np.arange(len(grafik_verisi)) / fps
)

# Yaklaşık bir saniyelik hareketli ortalama
pencere_boyutu = max(1, int(round(fps)))

for yol_adi in yol_bolgeleri:

    grafik_verisi[yol_adi + "_ORT"] = (
        grafik_verisi[yol_adi]
        .rolling(
            window=pencere_boyutu,
            center=True,
            min_periods=1
        )
        .mean()
    )


# ------------------------------------------------------------
# 6. BÜTÜN YOLLAR İÇİN ORTAK YOĞUNLUK EŞİKLERİ
# ------------------------------------------------------------

# Yol başına quantile kullanmak, çoğunlukla boş bir yolu
# tek araçla bile YÜKSEK gösterebildiği için kaldırılmıştır.
# Bütün yollar aynı fiziksel doluluk ölçeğinde değerlendirilir.
# Mevcut videodaki gözlemlere göre genişletilen aralık:
# %3 ve altı     -> DÜŞÜK
# %3 - %10 arası -> ORTA
# %10 üzeri      -> YÜKSEK
ortak_alt_esik = 3.00
ortak_ust_esik = 9.00

yol_esikleri = {
    yol_adi: {
        "alt": ortak_alt_esik,
        "ust": ortak_ust_esik
    }
    for yol_adi in yol_bolgeleri
}

print("\nBütün yollar için kullanılan ortak eşikler:")
print(f"DÜŞÜK : doluluk <= %{ortak_alt_esik:.2f}")
print(
    f"ORTA  : %{ortak_alt_esik:.2f} ile "
    f"%{ortak_ust_esik:.2f} arası"
)
print(f"YÜKSEK: doluluk > %{ortak_ust_esik:.2f}")


def yogunluk_sinifi(deger, alt_esik, ust_esik):

    # Yol doluluk değeri bütün video boyunca neredeyse
    # değişmiyorsa yanlış sınıflandırmayı engeller.
    if abs(ust_esik - alt_esik) < 0.001:
        return "DUSUK"

    if deger <= alt_esik:
        return "DUSUK"

    elif deger <= ust_esik:
        return "ORTA"

    else:
        return "YUKSEK"


# OpenCV renkleri BGR sırasındadır
renk_haritasi = {
    "DUSUK":  (0, 255, 0),     # Yeşil
    "ORTA":   (0, 255, 255),   # Sarı
    "YUKSEK": (0, 0, 255)      # Kırmızı
}


# ------------------------------------------------------------
# 7. RENKLİ SONUÇ VİDEOSUNU CPU İLE OLUŞTURMA
# ------------------------------------------------------------

print("\nRenkli sonuç videosu oluşturuluyor...")

cpu_baslangic = time.perf_counter()

giris_video = cv2.VideoCapture(video_yolu)

video_yazici = cv2.VideoWriter(
    gecici_video_yolu,
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (genislik, yukseklik)
)

kare_numarasi = 0

while True:

    basarili, kare = giris_video.read()

    if not basarili:
        break

    if kare_numarasi >= len(kare_kayitlari):
        break

    kayit = kare_kayitlari[kare_numarasi]
    renkli_katman = kare.copy()

    # Her yolu kendi yoğunluğuna göre renklendir
    for yol_adi, cokgen in yol_bolgeleri.items():

        doluluk = grafik_verisi.loc[
            kare_numarasi,
            yol_adi + "_ORT"
        ]

        alt_esik = yol_esikleri[yol_adi]["alt"]
        ust_esik = yol_esikleri[yol_adi]["ust"]

        sinif = yogunluk_sinifi(
            doluluk,
            alt_esik,
            ust_esik
        )

        renk = renk_haritasi[sinif]

        cv2.fillPoly(
            renkli_katman,
            [cokgen],
            renk
        )

    # Saydam renkli katmanı gerçek kareyle birleştir
    kare = cv2.addWeighted(
        renkli_katman,
        0.28,
        kare,
        0.72,
        0
    )

    # Yol sınırlarını ve bilgilerini çiz
    yazi_y = 35

    for yol_adi, cokgen in yol_bolgeleri.items():

        doluluk = grafik_verisi.loc[
            kare_numarasi,
            yol_adi + "_ORT"
        ]

        alt_esik = yol_esikleri[yol_adi]["alt"]
        ust_esik = yol_esikleri[yol_adi]["ust"]

        sinif = yogunluk_sinifi(
            doluluk,
            alt_esik,
            ust_esik
        )

        renk = renk_haritasi[sinif]

        cv2.polylines(
            kare,
            [cokgen],
            True,
            renk,
            3
        )

        bilgi = (
            f"{yol_adi}: {sinif} - %{doluluk:.2f}"
        )

        # Yazının görünmesi için siyah arka plan
        cv2.rectangle(
            kare,
            (10, yazi_y - 24),
            (470, yazi_y + 7),
            (0, 0, 0),
            -1
        )

        cv2.putText(
            kare,
            bilgi,
            (18, yazi_y),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.65,
            renk,
            2
        )

        yazi_y += 38

    # Araç kutularını ve ByteTrack numaralarını çiz
    for kutu, takip_id, sinif_no in zip(
        kayit["kutular"],
        kayit["takip_idleri"],
        kayit["siniflar"]
    ):

        x1, y1, x2, y2 = map(int, kutu)

        cv2.rectangle(
            kare,
            (x1, y1),
            (x2, y2),
            (255, 255, 255),
            2
        )

        if takip_id >= 0:
            arac_yazisi = f"ID:{takip_id}"
        else:
            arac_yazisi = "ARAC"

        cv2.putText(
            kare,
            arac_yazisi,
            (x1, max(20, y1 - 7)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.55,
            (255, 255, 255),
            2
        )

    # Genel bilgi
    cv2.putText(
        kare,
        f"Aktif arac: {kayit['aktif_arac']}",
        (genislik - 260, 35),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (255, 255, 255),
        2
    )

    video_yazici.write(kare)

    kare_numarasi += 1


giris_video.release()
video_yazici.release()

cpu_video_suresi = time.perf_counter() - cpu_baslangic

print(
    "CPU ile renkli video oluşturma süresi:",
    round(cpu_video_suresi, 2),
    "saniye"
)


# ------------------------------------------------------------
# 8. VİDEOYU H.264 FORMATINA DÖNÜŞTÜRME
# ------------------------------------------------------------

ffmpeg_baslangic = time.perf_counter()

subprocess.run(
    [
        "ffmpeg",
        "-y",
        "-i", gecici_video_yolu,
        "-vcodec", "libx264",
        "-pix_fmt", "yuv420p",
        "-movflags", "+faststart",
        sonuc_video_yolu
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    check=True
)

ffmpeg_suresi = time.perf_counter() - ffmpeg_baslangic


# ------------------------------------------------------------
# 9. SÜRE VE SONUÇ BİLGİLERİ
# ------------------------------------------------------------

toplam_sure = (
    gpu_suresi +
    cpu_video_suresi +
    ffmpeg_suresi
)

print("\n========== İŞLEM SÜRELERİ ==========")
print("YOLO + ByteTrack analiz süresi :", round(gpu_suresi, 2), "sn")
print("CPU görüntü oluşturma süresi   :", round(cpu_video_suresi, 2), "sn")
print("FFmpeg dönüştürme süresi       :", round(ffmpeg_suresi, 2), "sn")
print("Toplam işlem süresi            :", round(toplam_sure, 2), "sn")
print("Sonuç videosu                  :", sonuc_video_yolu)


# ------------------------------------------------------------
# 10. SONUÇ VİDEOSUNU GÖSTERME
# ------------------------------------------------------------

display(
    Video(
        sonuc_video_yolu,
        embed=True,
        width=900
    )
)